# Test Noise Passive Feedback

**Standalone 39 MHz bench-resonator measurement and feedback calibration.**

This notebook assumes the following wiring is already complete:

```text
OUT1 -> 30 dB pad -> air-gap pin -> coil -> 2.2 pF -> voltage follower
     -> ZFL-500LN+ -> IN1 directly, with no external 50 ohm termination

digitised IN1 -> FPGA unity monitor -> OUT2 -> 50 ohm spectrum analyser
```

No questionnaire blocks passive measurements. Safety decisions are made from measured waveform
levels, clipping, stationarity, spectrum shape, coefficient readback, resonance response and
feedback-off recovery. The configuration cell contains the assumed fixed hardware and can be
edited if the bench changes.

## How the notebook obtains the answer

| Stage | Calculation | What is calibrated |
|---|---|---|
| **Raw safety** | Full-rate IN1 samples give peak, RMS, DC offset, crest factor, histogram and clipping margin | Whether the amplified noise can safely enter the ADC |
| **Noise survey** | Hann-window FFTs are normalised by sample rate and window power to V²/Hz; independent spectra are averaged | ADC floor, powered-chain floor, resonator excess noise, spurs and averaging uncertainty |
| **Resonance fit** | `PSD = floor + slope + peak/[1 + 4((f-f0)/linewidth)^2]` | `f0`, linewidth, `Q=f0/linewidth`, covariance, residuals and drift |
| **Adaptive capture** | A broad acquisition finds the peak; later captures recenter on measured or predicted `f0` and keep at least 20 bins per linewidth | Span and resolution as Q and resonance frequency change |
| **Feedback gain** | OUT1 coefficients begin at one integer count and increase monotonically. Four phases separate damping and reactive effects | The effective loop gain through the unknown air-gap transfer |
| **Phase axes** | Q and frequency versus phase are fitted as orthogonal sinusoids | Regenerative, degenerative, positive-shift and negative-shift phases |
| **Gain ladder** | Opposite phases and feedback-off brackets distinguish response from drift; `K=(Q+ - Q-)/(Q+ + Q-)` | Linear interpolation envelopes for Q and frequency shift |
| **Recommendations** | Requested Q or shift is interpolated only between accepted measured points | A sealed, reviewable gain/phase setup—never extrapolated |

The loop is therefore calibrated from what it does to the resonator, not from a claimed coil
voltage. The 30 dB pad is nominal because its air-gap load is not 50 ohms. The exported record
always states `coil_voltage_directly_measured=False` and
`airgap_transfer_directly_measured=False`.

## OUT2 monitor meaning

OUT2 is a unity digital replay of IN1 at monitor shift zero. It is useful for a spectrum analyser,
but it is not an independent pickup: it contains ADC quantisation, FPGA scaling, DAC noise and
the output reconstruction response. Generator channel B is held at zero so it cannot contaminate
the trace. `feedback_off()` preserves a healthy campaign monitor; `full_shutdown()` disables it.

## Run order

1. Run configuration, imports and definitions.
2. Let the hardware-init and live-IN1 tests establish the safe state and ADC headroom.
3. Acquire the three noise conditions and the adaptive feedback-off baseline.
4. Enable and compare the unity OUT2 monitor.
5. Commission feedback from coefficient count one, then run phase and gain campaigns.
6. Seal calibration data before requesting or applying an operating point.


---
# How this notebook measures and applies feedback

The Red Pitaya contains **two parallel digital paths** after IN1. They must not be confused:

```text
                                      measurement only
                         +--> 39 MHz mixer --> CIC --> optional FIR --> DMA --> spectra
IN1 ADC, 125 MS/s -------+
                         +--> fractional-delay phase/gain block --> OUT1 --> coil
                                      active feedback
```

The CIC and FIR shape the spectra recorded by the notebook, but they are **not in the feedback
loop**. The active loop always runs at 125 MS/s. It combines the direct IN1 sample with a delayed
approximately-quadrature copy using coefficients `gain*cos(phase)` and `gain*sin(phase)`, then
sends the result to OUT1. Cable, amplifier, DAC/ADC, air-gap and coil phase are calibrated
empirically because the commanded FPGA phase is not the complete physical loop phase.

The initial measurement now captures about 1.95 MHz at supported decimation R=64, but only the
central **38.5–39.5 MHz** interval is searched. The wider raw margin is retained to expose filter
shape and edge artefacts. Before adaptive zoom, the notebook repeats the measurement with three
LO centres. A real resonator remains at the same absolute frequency; a CIC/FIR feature moves with
the LO. No fit may trigger recentring until this test passes.

Every active feedback campaign still begins at the smallest coefficient count and increases only
after clipping, broadband growth, self-oscillation, fit quality and feedback-off recovery pass.


In [ ]:
# Missing declarations warn but do not prevent feedback-off signal inspection.
TOPOLOGY = {
    'return_attenuation_db_nominal': 30.0,
    'input_attenuation_db': 0.0,
    'sense_capacitance_f': 2.2e-12,
    'voltage_follower_gain_db_nominal': -1.0,
    'zfl_model': 'ZFL-500LN+',
    'zfl_supply_v': 15.0,
    # IN1 is deliberately used in its native high-impedance mode. At 39 MHz,
    # its parallel input capacitance is important, so 1 Mohm is only the DC nominal.
    'in1_termination_ohm': 1.0e6,
    'in1_termination_location': 'NONE-NATIVE-HIGH-IMPEDANCE-IN1',
    'airgap_geometry_id': 'bench-airgap-39MHz-v1',
    'input_cable_id': 'CURRENT-IN1-CABLE',
    'feedback_cable_id': 'CURRENT-OUT1-CABLE',
    'resonance_frequency_hz': 39.0e6,
}
ANALYSER = {
    'model': 'CURRENT-50OHM-SPECTRUM-ANALYSER', 'input_impedance_ohm': 50.0,
    'maximum_safe_input_dbm': 20.0, 'reference_level_dbm': 10.0,
    'attenuation_db': 10.0, 'preamp_state': 'off', 'detector': 'RMS',
    'rbw_hz': 1000.0, 'vbw_hz': 1000.0, 'averages': 40,
    'cable_id': 'CURRENT-OUT2-CABLE', 'cable_loss_db': 0.0,
}

RP_LIMITS = {
    'sample_rate_hz': 125.0e6, 'adc_bits': 14,
    'input_impedance_ohm': 1.0e6, 'input_capacitance_f': 10e-12,
    'input_lv_full_scale_vpk': 1.0, 'input_lv_absolute_max_vpk': 6.0,
    'input_hv_full_scale_vpk': 20.0, 'input_hv_absolute_max_vpk': 30.0,
    'input_bandwidth_hz': 50.0e6, 'output_impedance_ohm': 50.0,
    'output_full_scale_vpk_into_50ohm': 1.0,
    'output_bandwidth_hz': 50.0e6, 'output_slew_rate_v_per_s': 200.0e6,
}
ZFL_LIMITS = {
    'frequency_min_hz': 0.1e6, 'frequency_max_hz': 500.0e6,
    'nominal_supply_v': 15.0, 'maximum_supply_v': 17.0,
    'gain_db_at_12v_typical': 24.0, 'gain_db_at_15v_typical': 28.0,
    'p1db_output_dbm_near_39mhz_typical': 8.0,
}
FS_CLK = RP_LIMITS['sample_rate_hz']
ADC_RANGE = 'LV'
ADC_OPERATING_GUARD_VPK = 0.80
ADC_RAW_VOLTS_PER_COUNT = RP_LIMITS['input_lv_full_scale_vpk'] / 8192.0
# DMA contains the 32-bit mixer/CIC output, not 14-bit ADC counts.  The RTL
# maps a full-scale ADC sine to complex baseband magnitude 2^29, independent
# of the power-of-two decimation.  Keeping these two scales separate prevents
# a harmless sub-millivolt CIC record being reported as tens of volts.
CIC_COUNTS_PER_ADC_VPK = float(1 << 29)
CIC_VOLTS_PER_COUNT = RP_LIMITS['input_lv_full_scale_vpk'] / CIC_COUNTS_PER_ADC_VPK
FIR_MAX_PASSBAND_BOOST_DB = 2.32
OUT1_COMMISSIONING_GUARD_VPK = 0.050
OUT1_OPERATING_GUARD_VPK = 0.80
OUT2_DECLARED_MAX_DBM = 10.0  # 1 Vpk sine into 50 ohm
OUT1_DC_OFFSET_LIMIT_V = 0.010
OUT1_TRANSIENT_LIMIT_VPK = 0.050
IN1_DC_OFFSET_WARNING_V = 0.050
IN1_DC_OFFSET_MAX_V = 0.100
OUT1_INITIAL_COEFFICIENT_COUNTS = (1, 2, 4, 8, 16, 32, 64, 128, 256)
OUT1_MAX_INITIAL_GAIN = 256.0 / 16384.0
CENTRE_HZ = TOPOLOGY['resonance_frequency_hz']
INITIAL_Q_ESTIMATE = 150.0
DISCOVERY_SEARCH_HALF_WIDTH_HZ = 0.50e6
DISCOVERY_SEARCH_SPAN_HZ = 2.0 * DISCOVERY_SEARCH_HALF_WIDTH_HZ
DISCOVERY_LO_OFFSETS_HZ = (-0.20e6, 0.0, 0.20e6)
FEEDBACK_MAX_EXPECTED_SHIFT_HZ = 1.0e6
MIN_ACQUISITION_DECIMATION_LOG2 = 6
MAX_ACQUISITION_DECIMATION_LOG2 = 12
FIR_MIN_DECIMATION_LOG2 = 8
DISCOVERY_Q_RANGE = (50.0, 1000.0)
DISCOVERY_EDGE_MARGIN_LINEWIDTHS = 1.5
MAX_RESPONSE_CORRECTION_DB = 12.0
BINS_PER_LINEWIDTH = 20
LINEWIDTHS_EACH_SIDE = 3.0
MAX_CAPTURE_SAMPLES = 1 << 20
PASSIVE_AVERAGES = 40
REFINED_BASELINE_AVERAGES = 120
SAFE_PROBE_AVERAGES = 120
LOCAL_FIT_LINEWIDTHS_EACH_SIDE = 1.25
LOCAL_Q_LOWER_FACTOR = 0.50
LOCAL_Q_UPPER_FACTOR = 2.00
SETTLE = 0.10
N_AVG = 64
DRIVE_MAX = 0.05
OUT2_MONITOR_SHIFT = 0
ELECTRODE_MAX_K = 0.30
CALIBRATION_MAX_AGE_S = 4 * 3600
BUILD_ID_EXPECTED = '0xFE76AE09'
CAMPAIGN_LABEL = 'test_noise_passive_feedback'
TRANSPORT_FACTORY = None       # tests inject MemoryTransport
CAPTURE_BACKEND = None          # tests inject complex passive records
CALIBRATION = None


---
# Analysis dependencies — no hardware access

**Purpose.** Import NumPy, SciPy and Matplotlib for PSD calculations, fitting and plots.

**Hardware.** None. **Expected result.** A short dependency/version message. **Failure meaning.**
A missing numerical package. **Safe to rerun.** Yes.


In [ ]:
import os, mmap, struct, time, json, copy, hashlib, math
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

print('Standalone numerical dependencies loaded.')


---
# Standalone safety, fitting, calibration, and recommendation library

**Purpose.** defines every pure-analysis and record-integrity function used below. It is an inline, reviewed copy owned by this notebook.

**Hardware.** none. **Wiring.** irrelevant. **Expected result.** none. **Pass.** the cell completes. **Failure.** a missing standard numerical dependency. **Safe to rerun.** yes.


In [ ]:
"""Safe, manual operating-point recommendations for electrode feedback.

This module contains no Red Pitaya register access.  It turns measurements from
the coil/electrode characterisation notebook into reviewable setup records.  A
notebook-specific adapter performs the eventual hardware write, after the
record and the current calibration have both been checked again.

The deliberately small public interface is:

    make_calibration(...)       validate and freeze measured calibration data
    feedback_q_setup(...)       recommend a Q-enhancement operating point
    feedback_shift_setup(...)   recommend a frequency-shift operating point
    validate_setup(...)         detect stale or modified setup records

Only interpolation between accepted measurements is allowed.  This is an
important safety property: a linear fit may describe the data nicely while
still being a dangerous basis for extrapolation towards self-oscillation.
"""
from __future__ import print_function

import copy
import hashlib
import json
import math
import time


SCHEMA_VERSION = 2


class CalibrationError(ValueError):
    """The measurements do not support a safe recommendation."""


class SetupError(ValueError):
    """A requested or saved setup is incomplete, stale, or unsafe."""


def _finite(name, value):
    value = float(value)
    if not math.isfinite(value):
        raise CalibrationError('%s must be finite' % name)
    return value


def _canonical(value):
    """Stable JSON representation used for calibration/setup fingerprints."""
    return json.dumps(value, sort_keys=True, separators=(',', ':'),
                      allow_nan=False).encode('utf-8')


def _digest(value):
    return hashlib.sha256(_canonical(value)).hexdigest()


def _without_digest(value):
    out = copy.deepcopy(value)
    out.pop('digest', None)
    return out


def _seal(value):
    value = copy.deepcopy(value)
    value['digest'] = _digest(_without_digest(value))
    return value


def _check_seal(value, label, error_type=SetupError):
    if not isinstance(value, dict) or not value.get('digest'):
        raise error_type('%s has no integrity fingerprint' % label)
    if value['digest'] != _digest(_without_digest(value)):
        raise error_type('%s changed after it was created' % label)


def _one_request(values):
    chosen = [(name, value) for name, value in values if value is not None]
    if len(chosen) > 1:
        raise SetupError('give exactly one request, not %s' %
                         ', '.join(name for name, _ in chosen))
    return chosen[0] if chosen else (None, None)


def _sorted_unique(points, xname, required):
    cleaned = []
    for number, point in enumerate(points):
        if not isinstance(point, dict):
            raise CalibrationError('%s point %d is not a dictionary' % (xname, number))
        row = {name: _finite('%s[%d].%s' % (xname, number, name), point[name])
               for name in required}
        row.update({name: value for name, value in point.items() if name not in row})
        if not bool(point.get('accepted', True)):
            continue
        cleaned.append(row)
    cleaned.sort(key=lambda row: row[xname])
    if len(cleaned) < 2:
        raise CalibrationError('%s needs at least two accepted measured points' % xname)
    xs = [row[xname] for row in cleaned]
    if any(b <= a for a, b in zip(xs[:-1], xs[1:])):
        raise CalibrationError('%s points must be unique' % xname)
    return cleaned


def make_calibration(baseline, q_points, shift_points, phases, evidence,
                     safety, build_id, created_utc=None):
    """Validate and freeze the evidence required by both setup helpers.

    ``q_points`` are accepted measured regenerative settings with gain, K and Q.
    ``shift_points`` use signed_gain so the two shift directions remain explicit.
    Measurements that failed their notebook quality gates must either be omitted
    or carry ``accepted=False``; they can never become recommendation anchors.
    """
    base = {name: _finite('baseline.%s' % name, baseline[name])
            for name in ('f0_hz', 'q0', 'linewidth_hz')}
    if base['f0_hz'] <= 0 or base['q0'] <= 0 or base['linewidth_hz'] <= 0:
        raise CalibrationError('baseline frequency, Q, and linewidth must be positive')

    phase_names = ('regenerative_deg', 'degenerative_deg',
                   'positive_shift_deg', 'negative_shift_deg')
    phase_data = {name: _finite('phases.%s' % name, phases[name]) % 360.0
                  for name in phase_names}
    phase_data['uncertainty_deg'] = _finite(
        'phases.uncertainty_deg', phases['uncertainty_deg'])
    if phase_data['uncertainty_deg'] <= 0:
        raise CalibrationError('phase uncertainty must be measured and positive')

    q_rows = _sorted_unique(q_points, 'gain',
                            ('gain', 'k', 'q', 'linewidth_hz',
                             'residual_shift_hz', 'uncertainty_q'))
    shift_rows = _sorted_unique(shift_points, 'signed_gain',
                                ('signed_gain', 'signed_k', 'shift_hz', 'q',
                                 'residual_q_change', 'uncertainty_hz'))
    if any(row['gain'] < 0 or row['k'] < 0 for row in q_rows):
        raise CalibrationError('Q-enhancement gain and K must be non-negative')

    max_k = _finite('safety.max_k', safety['max_k'])
    if not 0 < max_k <= 0.3:
        raise CalibrationError('bench-resonator max_k must be in (0, 0.3]')
    if any(row['k'] > max_k + 1e-12 for row in q_rows):
        raise CalibrationError('an accepted Q point exceeds the bench-resonator K limit')
    if any(abs(row.get('signed_k', 0.0)) > max_k + 1e-12 for row in shift_rows):
        raise CalibrationError('an accepted shift point exceeds the bench-resonator K limit')

    required_evidence = ('noise_floor_valid', 'baseline_valid', 'linearity_valid',
                         'phase_valid', 'gain_linearity_valid', 'drift_valid',
                         'acquisition_valid', 'analyser_valid',
                         'output_commissioning_valid', 'effective_feedback_valid',
                         'out1_dc_offset_valid', 'out2_monitor_valid')
    missing = [name for name in required_evidence if not bool(evidence.get(name, False))]
    if missing:
        raise CalibrationError('required calibration evidence failed: %s' %
                               ', '.join(missing))
    required_hardware = ('topology_digest', 'return_attenuation_db_nominal',
                         'input_attenuation_db', 'sense_capacitance_f',
                         'in1_termination_ohm', 'analyser_impedance_ohm',
                         'airgap_geometry_id')
    missing = [name for name in required_hardware if name not in safety]
    if missing:
        raise CalibrationError(
            'Calibration lacks concrete hardware identity fields: %s. Rebuild the '
            'campaign record from the current notebook configuration.' % ', '.join(missing))
    expected_hardware = {'return_attenuation_db_nominal': 30.0,
                         'input_attenuation_db': 0.0,
                         'sense_capacitance_f': 2.2e-12,
                         'in1_termination_ohm': 1.0e6,
                         'analyser_impedance_ohm': 50.0}
    for name, expected_value in expected_hardware.items():
        actual = _finite('safety.' + name, safety[name])
        if not math.isclose(actual, expected_value, rel_tol=1e-9):
            raise CalibrationError(
                'Calibration hardware mismatch: %s=%.9g, required %.9g for this '
                'campaign. Do not reuse calibration after hardware changes.' %
                (name, actual, expected_value))
    if not str(safety['topology_digest']).strip() or not str(safety['airgap_geometry_id']).strip():
        raise CalibrationError('Topology digest and air-gap identity must be non-empty strings.')


    calibration = {
        'schema_version': SCHEMA_VERSION,
        'created_utc': created_utc or time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'build_id': str(build_id),
        'baseline': base,
        'phases': phase_data,
        'q_points': q_rows,
        'shift_points': shift_rows,
        'evidence': copy.deepcopy(evidence),
        'safety': copy.deepcopy(safety),
        'coil_voltage_directly_measured': False,
        'airgap_transfer_directly_measured': False,
        'effective_loop_status': 'closed_loop_inference_only',
    }
    return _seal(calibration)


def validate_calibration(calibration):
    _check_seal(calibration, 'calibration', CalibrationError)
    if calibration.get('schema_version') != SCHEMA_VERSION:
        raise CalibrationError('unsupported calibration schema')
    return calibration


def _bracket(rows, field, target):
    """Return two measured rows surrounding target; never extrapolate."""
    target = float(target)
    lo, hi = rows[0][field], rows[-1][field]
    if target < lo - 1e-12 or target > hi + 1e-12:
        raise SetupError('requested %s %.6g is outside measured range %.6g..%.6g' %
                         (field, target, lo, hi))
    for left, right in zip(rows[:-1], rows[1:]):
        if left[field] <= target <= right[field]:
            return left, right
    return rows[-2], rows[-1]


def _interpolate(rows, field, target, names):
    rows = sorted(rows, key=lambda row: row[field])
    left, right = _bracket(rows, field, target)
    span = right[field] - left[field]
    weight = 0.0 if span == 0 else (float(target) - left[field]) / span
    result = {name: float(left[name]) + weight * (float(right[name]) - float(left[name]))
              for name in names}
    result[field] = float(target)
    return result, [left[field], right[field]], weight


def _q_target(calibration, target_q, q_ratio, target_k, gain):
    rows, base = calibration['q_points'], calibration['baseline']
    name, value = _one_request((('target_q', target_q), ('q_ratio', q_ratio),
                                ('target_k', target_k), ('gain', gain)))
    if name is None:
        # The first non-zero accepted measurement is deliberately preferred.
        # A notebook may display every rung, but an omitted target must never
        # silently select the operating point closest to instability.
        conservative = next((row for row in rows if row['gain'] > 0), rows[0])
        return 'recommended_conservative', conservative['gain'], 'gain'
    value = float(value)
    if name == 'target_q':
        return name, value, 'q'
    if name == 'q_ratio':
        if value <= 0:
            raise SetupError('q_ratio must be positive')
        return name, value * base['q0'], 'q'
    return name, value, 'k' if name == 'target_k' else 'gain'


def _new_setup(calibration, mode, request, resolved, bracket, weight):
    setup = {
        'schema_version': SCHEMA_VERSION,
        'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'mode': mode,
        'request': request,
        'calibration_digest': calibration['digest'],
        'build_id': calibration['build_id'],
        'baseline': copy.deepcopy(calibration['baseline']),
        'resolved': resolved,
        'interpolation_bracket': list(bracket),
        'interpolation_weight': float(weight),
        'safety': {'max_k': float(calibration['safety']['max_k']),
                   'interpolation_only': True},
        'validation': None,
        'coil_voltage_directly_measured': False,
        'airgap_transfer_directly_measured': False,
    }
    return _seal(setup)


def feedback_q_setup(calibration, target_q=None, q_ratio=None,
                     target_k=None, gain=None):
    """Return a reviewable Q-enhancement setup; never touch hardware."""
    validate_calibration(calibration)
    request_name, target, field = _q_target(
        calibration, target_q, q_ratio, target_k, gain)
    values = ('gain', 'k', 'q', 'linewidth_hz', 'residual_shift_hz', 'uncertainty_q')
    resolved, bracket, weight = _interpolate(
        calibration['q_points'], field, target, values)
    if resolved['k'] > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('recommended K exceeds the bench-resonator limit')
    resolved.update({
        'phase_deg': calibration['phases']['regenerative_deg'],
        'phase_uncertainty_deg': calibration['phases']['uncertainty_deg'],
        'stability_margin': 1.0 - resolved['k'],
        'predicted_f0_hz': calibration['baseline']['f0_hz'] +
                           resolved['residual_shift_hz'],
    })
    return _new_setup(calibration, 'q_enhancement',
                      {request_name: target}, resolved, bracket, weight)


def feedback_shift_setup(calibration, shift_hz=None, shift_linewidths=None,
                         signed_gain=None, target_k=None, direction=None):
    """Return one or two reviewable frequency-shift setups; no hardware access.

    With no request, a dictionary containing conservative positive and negative
    measured endpoints is returned.  ``target_k`` needs direction +1 or -1.
    """
    validate_calibration(calibration)
    name, value = _one_request((('shift_hz', shift_hz),
                                ('shift_linewidths', shift_linewidths),
                                ('signed_gain', signed_gain),
                                ('target_k', target_k)))
    rows = calibration['shift_points']
    if name is None:
        # Present the smallest measured non-zero setting in each direction.
        # Larger endpoints remain available through an explicit request.
        negative = max((row for row in rows if row['signed_gain'] < 0),
                       key=lambda row: row['signed_gain'])
        positive = min((row for row in rows if row['signed_gain'] > 0),
                       key=lambda row: row['signed_gain'])
        return {'negative': feedback_shift_setup(calibration,
                                                 signed_gain=negative['signed_gain']),
                'positive': feedback_shift_setup(calibration,
                                                 signed_gain=positive['signed_gain'])}
    value = float(value)
    if name == 'shift_linewidths':
        target, field = value * calibration['baseline']['linewidth_hz'], 'shift_hz'
    elif name == 'shift_hz':
        target, field = value, 'shift_hz'
    elif name == 'signed_gain':
        target, field = value, 'signed_gain'
    else:
        if direction not in (-1, 1):
            raise SetupError('target_k requires direction=-1 or direction=+1')
        target = float(direction) * value
        # Shift points carry measured vector K when available.  Keep its sign so
        # the two physical frequency directions cannot be confused.
        field = 'signed_k'
        if not all(field in row for row in rows):
            raise SetupError('shift calibration has no signed_k measurements')

    values = ('signed_gain', 'signed_k', 'shift_hz', 'q',
              'residual_q_change', 'uncertainty_hz')
    resolved, bracket, weight = _interpolate(rows, field, target, values)
    resolved['k'] = abs(resolved['signed_k'])
    if resolved['k'] > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('recommended K exceeds the bench-resonator limit')
    positive = resolved['shift_hz'] >= 0
    resolved.update({
        'phase_deg': calibration['phases'][
            'positive_shift_deg' if positive else 'negative_shift_deg'],
        'phase_uncertainty_deg': calibration['phases']['uncertainty_deg'],
        'stability_margin': 1.0 - resolved['k'],
        'predicted_f0_hz': calibration['baseline']['f0_hz'] + resolved['shift_hz'],
        'predicted_linewidth_hz': (calibration['baseline']['linewidth_hz'] *
                                   calibration['baseline']['q0'] /
                                   max(resolved['q'], 1e-30)),
    })
    return _new_setup(calibration, 'frequency_shift', {name: value},
                      resolved, bracket, weight)


def validate_setup(setup, calibration, current_build_id=None, max_age_s=None,
                   now_s=None):
    """Revalidate a setup immediately before a notebook applies it."""
    validate_calibration(calibration)
    _check_seal(setup, 'setup')
    if setup.get('calibration_digest') != calibration['digest']:
        raise SetupError('setup belongs to a different or stale calibration')
    if current_build_id is not None and str(current_build_id) != str(setup['build_id']):
        raise SetupError('current FPGA build does not match the calibration')
    if setup['resolved'].get('k', 0.0) > calibration['safety']['max_k'] + 1e-12:
        raise SetupError('setup exceeds the current bench-resonator K limit')
    if max_age_s is not None:
        # The notebook supplies age from its monotonic laboratory timestamp;
        # parsing wall-clock strings here would be timezone-dependent on old OSes.
        age = float(now_s if now_s is not None else time.time()) - float(
            calibration['evidence']['created_epoch_s'])
        if age < 0 or age > float(max_age_s):
            raise SetupError('calibration is stale (age %.0f s)' % age)
    return True


# ---------------------------------------------------------------------------
# Passive measurement and preflight helpers
# ---------------------------------------------------------------------------

def validate_topology(config):
    """Validate concrete bench values; no operator confirmation flags are used."""
    required = ('return_attenuation_db_nominal', 'input_attenuation_db',
                'sense_capacitance_f', 'voltage_follower_gain_db_nominal',
                'zfl_model', 'zfl_supply_v', 'in1_termination_ohm',
                'in1_termination_location', 'airgap_geometry_id',
                'input_cable_id', 'feedback_cable_id', 'resonance_frequency_hz')
    missing = [name for name in required if name not in config or config[name] in (None, '')]
    if missing:
        raise CalibrationError(
            'Bench topology is incomplete. Missing concrete values: %s. '
            'Record component values and identifiers; True/False confirmations are not used.' %
            ', '.join(missing))
    frequency_hz = _finite('resonance_frequency_hz', config['resonance_frequency_hz'])
    if not 0.1e6 <= frequency_hz <= 50.0e6:
        raise CalibrationError(
            'Configured resonance %.6g Hz is outside the common 0.1--50 MHz '
            'ZFL/Red-Pitaya measurement band.' % frequency_hz)
    expected = {'return_attenuation_db_nominal': 30.0,
                'input_attenuation_db': 0.0,
                'sense_capacitance_f': 2.2e-12,
                'in1_termination_ohm': 1.0e6}
    for name, expected_value in expected.items():
        actual = _finite(name, config[name])
        if not math.isclose(actual, expected_value, rel_tol=1e-9):
            raise CalibrationError(
                '%s is %.9g but this calibration campaign requires %.9g. '
                'Changing a pad, capacitor or termination starts a new campaign.' %
                (name, actual, expected_value))
    if str(config['airgap_geometry_id']).startswith(('RECORD-', 'CURRENT-')):
        raise CalibrationError(
            'airgap_geometry_id is still a placeholder (%r). Use a stable physical '
            'identifier before sealing a calibration.' % config['airgap_geometry_id'])
    return _seal({'schema_version': SCHEMA_VERSION,
                  'kind': 'test_noise_passive_feedback_topology',
                  'config': copy.deepcopy(config)})

def analyser_configuration_fingerprint(config):
    """Fingerprint all analyser/cabling settings that must remain unchanged."""
    required = ('model', 'input_impedance_ohm', 'maximum_safe_input_dbm',
                'reference_level_dbm', 'attenuation_db', 'preamp_state',
                'detector', 'rbw_hz', 'vbw_hz', 'averages', 'cable_id',
                'cable_loss_db')
    missing = [name for name in required if name not in config]
    if missing:
        raise CalibrationError('analyser configuration is incomplete: %s' %
                               ', '.join(missing))
    if float(config['input_impedance_ohm']) != 50.0:
        raise CalibrationError('spectrum analyser input must be 50 ohm')
    return _digest({name: config[name] for name in required})


def passive_welch_spectrum(records, sample_rate_hz, window='hann'):
    """Return a correctly normalized one-sided PSD from passive time records.

    ``records`` is a two-dimensional array (one row per independent capture).
    The returned metadata exposes bin width, equivalent-noise bandwidth and
    averaging uncertainty instead of hiding these quantities in a plot.
    """
    import numpy as np
    data = np.asarray(records, dtype=float)
    if data.ndim != 2 or data.shape[0] < 2 or data.shape[1] < 16:
        raise CalibrationError('need at least two passive records of 16 samples')
    if not np.all(np.isfinite(data)) or float(sample_rate_hz) <= 0:
        raise CalibrationError('records and sample rate must be finite')
    n = data.shape[1]
    if window == 'hann':
        weights = np.hanning(n)
    elif window == 'rectangular':
        weights = np.ones(n)
    else:
        raise CalibrationError('window must be hann or rectangular')
    detrended = data - np.mean(data, axis=1, keepdims=True)
    fft = np.fft.rfft(detrended * weights[None, :], axis=1)
    scale = float(sample_rate_hz) * float(np.sum(weights ** 2))
    individual = np.abs(fft) ** 2 / scale
    if n % 2 == 0:
        individual[:, 1:-1] *= 2.0
    else:
        individual[:, 1:] *= 2.0
    mean_psd = np.mean(individual, axis=0)
    stderr = np.std(individual, axis=0, ddof=1) / math.sqrt(data.shape[0])
    bin_hz = float(sample_rate_hz) / n
    enbw_hz = float(sample_rate_hz) * float(np.sum(weights ** 2)) / float(np.sum(weights) ** 2)
    return {'frequency_hz': np.fft.rfftfreq(n, 1.0 / float(sample_rate_hz)),
            'psd_v2_per_hz': mean_psd, 'stderr_v2_per_hz': stderr,
            'individual_psd_v2_per_hz': individual,
            'bin_hz': bin_hz, 'enbw_hz': enbw_hz,
            'averages': int(data.shape[0]),
            'relative_standard_uncertainty': 1.0 / math.sqrt(data.shape[0])}


def passive_iq_spectrum(records, sample_rate_hz, centre_hz, window='hann'):
    """Return a centered two-sided PSD for complex down-converted records."""
    import numpy as np
    data = np.asarray(records, dtype=complex)
    if data.ndim != 2 or data.shape[0] < 2 or data.shape[1] < 16:
        raise CalibrationError('need at least two complex passive records')
    if not np.all(np.isfinite(data)) or float(sample_rate_hz) <= 0:
        raise CalibrationError('records and sample rate must be finite')
    n = data.shape[1]
    weights = np.hanning(n) if window == 'hann' else np.ones(n)
    data = data - np.mean(data, axis=1, keepdims=True)
    transformed = np.fft.fftshift(np.fft.fft(data * weights[None, :], axis=1), axes=1)
    individual = np.abs(transformed) ** 2 / (float(sample_rate_hz) * np.sum(weights ** 2))
    return {'frequency_hz': float(centre_hz) + np.fft.fftshift(np.fft.fftfreq(n, 1.0 / float(sample_rate_hz))),
            'psd_v2_per_hz': np.mean(individual, axis=0),
            'stderr_v2_per_hz': np.std(individual, axis=0, ddof=1) / math.sqrt(data.shape[0]),
            'individual_psd_v2_per_hz': individual,
            'bin_hz': float(sample_rate_hz) / n,
            'enbw_hz': float(sample_rate_hz) * np.sum(weights ** 2) / np.sum(weights) ** 2,
            'averages': int(data.shape[0]),
            'relative_standard_uncertainty': 1.0 / math.sqrt(data.shape[0])}


def fit_passive_resonator(frequency_hz, psd):
    """Fit the bench resonator's broad passive-noise peak over a sloping floor."""
    import numpy as np
    from scipy.optimize import least_squares
    f = np.asarray(frequency_hz, dtype=float)
    y = np.asarray(psd, dtype=float)
    if f.ndim != 1 or y.shape != f.shape or len(f) < 20 or np.any(y <= 0):
        raise CalibrationError('fit requires matching positive 1D spectrum arrays')
    order = np.argsort(f); f, y = f[order], y[order]
    span = float(f[-1] - f[0]); centre = 0.5 * (f[0] + f[-1])
    spacing = float(np.median(np.diff(f)))
    x = (f - centre) / max(span, np.finfo(float).tiny)
    floor = float(np.percentile(y, 20)); peak_i = int(np.argmax(y))
    amplitude = max(float(y[peak_i] - floor), np.finfo(float).tiny)
    linewidth0 = max(span / 20.0, spacing * 4.0)

    def model(parameters):
        b0, b1, peak, f0, linewidth = parameters
        return b0 + b1 * x + peak / (1.0 + 4.0 * ((f - f0) / linewidth) ** 2)

    initial = [floor, 0.0, amplitude, float(f[peak_i]), linewidth0]
    lower = [0.0, -10 * max(floor, amplitude), 0.0, f[0], spacing]
    upper = [10 * max(y), 10 * max(floor, amplitude), 20 * max(y), f[-1], span]
    scale = np.maximum(np.sqrt(y), np.sqrt(np.median(y)) * 0.2)
    result = least_squares(lambda values: (model(values) - y) / scale,
                           initial, bounds=(lower, upper), max_nfev=20000)
    if not result.success:
        raise CalibrationError('passive resonator fit did not converge')
    fitted = model(result.x); residual = y - fitted
    dof = max(1, len(y) - len(result.x)); rss = float(np.sum(residual ** 2))
    try:
        covariance = np.linalg.inv(result.jac.T.dot(result.jac)) * (2 * result.cost / dof)
        uncertainty = np.sqrt(np.maximum(np.diag(covariance), 0.0))
    except np.linalg.LinAlgError:
        covariance = np.full((len(result.x), len(result.x)), np.nan)
        uncertainty = np.full(len(result.x), np.nan)
    f0, linewidth = float(result.x[3]), float(result.x[4])
    return {'accepted': bool(np.isfinite(uncertainty[3]) and linewidth > 0),
            'resonator_state': 'passive_noise', 'f0_hz': f0,
            'linewidth_hz': linewidth, 'q': f0 / linewidth,
            'noise_floor': float(result.x[0]), 'slope': float(result.x[1]),
            'resonator_amplitude': float(result.x[2]), 'model': fitted,
            'residual': residual, 'rss': rss, 'reduced_residual': rss / dof,
            'covariance': covariance,
            'f0_uncertainty_hz': float(uncertainty[3]),
            'linewidth_uncertainty_hz': float(uncertainty[4])}

def assess_passive_records(records, adc_limit_vpk, clipping_fraction=0.98,
                           stationarity_cv_limit=0.20):
    """Check ADC headroom, clipping, crest factor and record stationarity."""
    import numpy as np
    data = np.asarray(records, dtype=float)
    if data.ndim != 2 or data.size == 0:
        raise CalibrationError('passive records must be a non-empty 2D array')
    limit = abs(_finite('adc_limit_vpk', adc_limit_vpk))
    peak = float(np.max(np.abs(data)))
    rms_by_record = np.sqrt(np.mean((data - np.mean(data, axis=1, keepdims=True)) ** 2,
                                    axis=1))
    mean_rms = float(np.mean(rms_by_record))
    cv = float(np.std(rms_by_record, ddof=1) / mean_rms) if len(rms_by_record) > 1 and mean_rms else float('inf')
    clipped = bool(peak >= clipping_fraction * limit)
    stationary = bool(cv <= stationarity_cv_limit)
    return {'accepted': bool(not clipped and stationary), 'peak_v': peak,
            'headroom_v': limit - peak,
            'crest_factor': peak / mean_rms if mean_rms else float('inf'),
            'rms_v': mean_rms, 'rms_cv': cv, 'clipped': clipped,
            'stationary': stationary}


def detect_self_oscillation(frequency_hz, psd, baseline_psd,
                            carrier_ratio_limit=20.0,
                            broadband_ratio_limit=3.0):
    """Detect a narrow carrier or broadband power increase over baseline."""
    import numpy as np
    values = np.asarray(psd, dtype=float)
    baseline = np.asarray(baseline_psd, dtype=float)
    if values.shape != baseline.shape or values.size < 5:
        raise CalibrationError('current and baseline PSD arrays must match')
    floor = np.maximum(baseline, np.finfo(float).tiny)
    ratio = values / floor
    carrier_ratio = float(np.max(ratio))
    broadband_ratio = float(np.sum(values) / max(np.sum(baseline), np.finfo(float).tiny))
    return {'unsafe': bool(carrier_ratio > carrier_ratio_limit or
                           broadband_ratio > broadband_ratio_limit),
            'carrier_ratio': carrier_ratio,
            'broadband_ratio': broadband_ratio,
            'carrier_frequency_hz': float(np.asarray(frequency_hz)[int(np.argmax(ratio))])}


def paired_k(q_plus, q_minus):
    """Compute K from closely spaced opposite-phase Q measurements."""
    q_plus = _finite('q_plus', q_plus)
    q_minus = _finite('q_minus', q_minus)
    if q_plus <= 0 or q_minus <= 0:
        raise CalibrationError('paired Q measurements must be positive')
    return (q_plus - q_minus) / (q_plus + q_minus)



def plan_passive_acquisition(f0_hz, q_estimate, expected_shift_hz=0.0,
                             clock_hz=125e6, min_decimation_log2=6,
                             max_decimation_log2=12,
                             bins_per_linewidth=20,
                             linewidths_each_side=3.0,
                             shift_guard_fraction=0.25,
                             centre_on_expected_shift=False,
                             usable_band_fraction=0.80,
                             min_samples=2048, max_samples=1 << 20):
    """Plan a supported complex-baseband capture without hiding truncation.

    The FPGA acquisition register is qualified only for DEC_LOG2 6..12.  The
    complete FFT interval is not automatically a trustworthy fitting interval:
    CIC gain falls with distance from the LO and the FIR is bypassed below
    DEC_LOG2=8.  Callers must separately apply an explicit absolute-frequency
    search mask and a measured or modelled acquisition-response correction.
    """
    f0 = _finite('f0_hz', f0_hz)
    q = _finite('q_estimate', q_estimate)
    shift = abs(_finite('expected_shift_hz', expected_shift_hz))
    minimum = int(min_decimation_log2)
    maximum = int(max_decimation_log2)
    if minimum < 6 or maximum > 12 or minimum > maximum:
        raise CalibrationError(
            'Unsupported acquisition decimation request %d..%d. The qualified '
            'FPGA register range is DEC_LOG2=6..12 (R=64..4096). Do not use '
            'DEC_LOG2=4 or 5: those settings were not commissioned and expose '
            'a very broad uncorrected CIC response.' % (minimum, maximum))
    if f0 <= 0 or q <= 0 or bins_per_linewidth < 5 or linewidths_each_side < 1.5:
        raise CalibrationError(
            'Invalid acquisition request: f0=%.9g Hz, Q=%.9g, bins/linewidth=%r, '
            'linewidths/side=%r. Frequency and Q must be positive, at least five '
            'bins/linewidth and 1.5 linewidths/side are required.' %
            (f0, q, bins_per_linewidth, linewidths_each_side))
    linewidth = f0 / q
    shift_allowance = shift_guard_fraction * shift if centre_on_expected_shift else shift
    required_usable_span = 2.0 * (shift_allowance + linewidths_each_side * linewidth)
    chosen_decimation = None
    for decimation in range(maximum, minimum - 1, -1):
        sample_rate = float(clock_hz) / (1 << decimation)
        if usable_band_fraction * sample_rate >= required_usable_span:
            chosen_decimation = decimation
            break
    if chosen_decimation is None:
        largest_span = usable_band_fraction * float(clock_hz) / (1 << minimum)
        raise CalibrationError(
            'The requested usable span is %.6g Hz, but the largest qualified '
            'capture provides %.6g Hz at DEC_LOG2=%d. Reduce the shift allowance '
            'or linewidth coverage, or use overlapping LO tiles; the notebook '
            'will not silently use unsupported decimation or truncate a resonance.' %
            (required_usable_span, largest_span, minimum))
    sample_rate = float(clock_hz) / (1 << chosen_decimation)
    target_bin = linewidth / float(bins_per_linewidth)
    needed = max(int(min_samples), int(math.ceil(sample_rate / target_bin)))
    samples = 1 << int(math.ceil(math.log(max(needed, 2), 2)))
    if samples > int(max_samples):
        raise CalibrationError(
            'Q %.6g at %.9g Hz requires %d samples for %.1f bins/linewidth, '
            'above the safe limit %d. Increase the capture limit only after DMA '
            'memory bounds and acquisition time have been reviewed.' %
            (q, f0, samples, bins_per_linewidth, max_samples))
    bin_hz = sample_rate / samples
    return {'centre_hz': f0 + (float(expected_shift_hz)
                               if centre_on_expected_shift else 0.0),
            'reference_f0_hz': f0, 'q_estimate': q,
            'linewidth_hz': linewidth,
            'expected_shift_hz': float(expected_shift_hz),
            'centre_on_expected_shift': bool(centre_on_expected_shift),
            'decimation_log2': chosen_decimation,
            'sample_rate_hz': sample_rate, 'samples_per_record': samples,
            'bin_hz': bin_hz, 'bins_per_linewidth': linewidth / bin_hz,
            'full_span_hz': sample_rate,
            'usable_span_hz': usable_band_fraction * sample_rate,
            'record_duration_s': samples / sample_rate,
            'usable_band_fraction': float(usable_band_fraction),
            'fir_expected_state': ('active' if chosen_decimation >= 8 else 'bypassed'),
            'qualified_decimation_range': [6, 12]}


---
# 2 — Review the declared bench values and calculate the discovery capture

**What this cell does.** It prints the fixed bench topology and calculates the first acquisition.
The raw capture is intentionally wider than the accepted 1 MHz search interval so filter edges
remain visible, but resonance finding is restricted to 38.5–39.5 MHz.

**Why it is needed.** The earlier run used unsupported decimation R=16 and interpreted the broad
CIC passband as a resonator. This cell now enforces the FPGA-qualified range R=64…4096 and reports
whether the FIR is expected to be active or bypassed.

**Hardware and wiring.** This cell performs calculations only. It does not read or write the Red
Pitaya. The declared topology must match the physical 30 dB return pad, air gap, 2.2 pF sensing
capacitor, follower, ZFL and direct high-impedance IN1 connection (no input pad or terminator).

**Expected result and pass criteria.** The discovery plan should select DEC_LOG2=6 (R=64), about
1.953 MHz raw span, and identify a 1 MHz accepted search interval. Any unsupported decimation or
insufficient bandwidth produces an explanatory error rather than a truncated spectrum.

**Failure meaning and rerun safety.** A failure is a planning/configuration problem, not evidence
about the resonator. Correct the declared frequency/Q/search settings and rerun; no hardware was
touched.


In [ ]:
def configuration_report():
    nominal_linewidth_hz = CENTRE_HZ / INITIAL_Q_ESTIMATE
    print('Assumed bench topology (change the configuration cell if hardware changes)')
    print('  OUT1 -> 30 dB -> air gap -> coil -> 2.2 pF -> follower -> ZFL')
    print('       -> IN1 directly in native high-impedance mode (no pad or terminator)')
    print('  IN1 digital unity monitor -> OUT2 -> 50 ohm spectrum analyser')
    print('  nominal centre %.3f MHz, Q %.1f, linewidth %.1f kHz' %
          (CENTRE_HZ / 1e6, INITIAL_Q_ESTIMATE, nominal_linewidth_hz / 1e3))
    print('  ADC operating guard %.3f Vpk; OUT2 initial shift = 0 (unity)' %
          ADC_OPERATING_GUARD_VPK)
    print('  discovery averages %d; refined/probe averages %d/%d' %
          (PASSIVE_AVERAGES, REFINED_BASELINE_AVERAGES, SAFE_PROBE_AVERAGES))
    print('  local fit uses +/- %.2f discovery linewidths and Q range %.2f..%.2f x discovery Q' %
          (LOCAL_FIT_LINEWIDTHS_EACH_SIDE, LOCAL_Q_LOWER_FACTOR, LOCAL_Q_UPPER_FACTOR))
    return {
        'topology_digest': _digest(TOPOLOGY),
        'analyser_fingerprint': analyser_configuration_fingerprint(ANALYSER),
        'passive_measurement_mode': 'ready_for_live_safety_tests',
    }

CONFIGURATION_REPORT = configuration_report()
TOPOLOGY_RECORD = _seal({'schema_version': SCHEMA_VERSION,
                         'kind': 'bench_loop_topology',
                         'config': copy.deepcopy(TOPOLOGY)})
ANALYSER_FINGERPRINT = analyser_configuration_fingerprint(ANALYSER)
INITIAL_ACQUISITION_PLAN = plan_passive_acquisition(
    CENTRE_HZ, INITIAL_Q_ESTIMATE, 0.0,
    min_decimation_log2=MIN_ACQUISITION_DECIMATION_LOG2,
    max_decimation_log2=MAX_ACQUISITION_DECIMATION_LOG2,
    bins_per_linewidth=BINS_PER_LINEWIDTH,
    linewidths_each_side=LINEWIDTHS_EACH_SIDE,
    max_samples=MAX_CAPTURE_SAMPLES)
print('Initial discovery capture: %.1f kHz raw span, %.1f kHz planner span, %.1f Hz bins, DEC_LOG2=%d' %
      (INITIAL_ACQUISITION_PLAN['full_span_hz'] / 1e3,
       INITIAL_ACQUISITION_PLAN['usable_span_hz'] / 1e3,
       INITIAL_ACQUISITION_PLAN['bin_hz'],
       INITIAL_ACQUISITION_PLAN['decimation_log2']))
print('Accepted discovery search: %.3f to %.3f MHz; FIR expected %s' %
      ((CENTRE_HZ-DISCOVERY_SEARCH_HALF_WIDTH_HZ)/1e6,
       (CENTRE_HZ+DISCOVERY_SEARCH_HALF_WIDTH_HZ)/1e6,
       INITIAL_ACQUISITION_PLAN['fir_expected_state']))


---
# Standalone register transport and plotting definitions

**Purpose.** defines the real and mock register transports and plotting compatibility helpers.

**Hardware.** no hardware is opened until a class is instantiated. **Wiring.** unchanged. **Pass.** definitions complete without output. **Failure.** Python dependency/version problem. **Safe to rerun.** yes.


In [ ]:
class MemoryTransport(object):
    """Byte-addressable mock register block used by notebook tests."""
    def __init__(self, base, span):
        self.base = base
        self.m = bytearray(span)
        self.fd = None
        self.fail_reads = frozenset()
        self.fail_writes = frozenset()
        self.write_log = []

    def close(self):
        pass


class DevMemTransport(object):
    """Real Red Pitaya register mapping. Construction is the hardware boundary."""
    def __init__(self, base, span):
        self.base = base
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m = mmap.mmap(self.fd, span, offset=base)

    def close(self):
        self.m.close()
        os.close(self.fd)


class Bus(object):
    """One register block with an injectable real or memory-backed transport."""
    def __init__(self, base, span=0x1000, transport_factory=None):
        factory = transport_factory or TRANSPORT_FACTORY or DevMemTransport
        self.transport = factory(base, span)
        self.base = base
        self.fd = self.transport.fd
        self.m = self.transport.m

    def w(self, off, val):
        if off in getattr(self.transport, 'fail_writes', frozenset()):
            raise IOError('injected register-write failure at 0x%X' % off)
        self.m[off:off + 4] = struct.pack('<I', int(val) & 0xFFFFFFFF)
        if hasattr(self.transport, 'write_log'):
            self.transport.write_log.append((off, int(val) & 0xFFFFFFFF))

    def r(self, off):
        if off in getattr(self.transport, 'fail_reads', frozenset()):
            raise IOError('injected register-read failure at 0x%X' % off)
        return struct.unpack('<I', bytes(self.m[off:off + 4]))[0]

    def rs(self, off):
        value = self.r(off)
        return value - (1 << 32) if value & 0x80000000 else value

    def close(self):
        self.transport.close()


def shutdown_devices(generator, phase_shift, acquisition):
    """Attempt every shutdown action and verify zero feedback coefficients."""
    errors = []
    for label, action in (('generator', generator.off),
                          ('feedback', phase_shift.off),
                          ('acquisition', acquisition.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        readback = phase_shift.readback()
        if int(readback['ci']) != 0 or int(readback['cq']) != 0:
            errors.append('feedback coefficient readback is not zero: %r' %
                          (readback,))
    except BaseException as exc:
        errors.append('feedback readback: %s' % exc)
    if errors:
        raise RuntimeError('safe shutdown could not be verified; physically '
                           'disconnect OUT1: ' + '; '.join(errors))
    return True


# =============================================================================
#  PLOTTING COMPATIBILITY  -  this board runs matplotlib 1.x
# =============================================================================
#  The 'C0'..'C9' colour shorthand arrived in matplotlib 2.0. On 1.x it is not
#  rejected when you pass it - it is accepted, stored, and then raises
#
#      ValueError: to_rgba: Invalid rgba arg "C3"
#
#  from inside the draw, at which point the figure is already built. What you see
#  is a traceback followed by a bare <matplotlib.figure.Figure at 0x...> instead
#  of a plot, and every OTHER cell still runs, so it looks like a plotting
#  nuisance rather than a version problem. Every colour in this notebook is
#  therefore written out in full.
CLR_BLUE   = '#1f77b4'
CLR_ORANGE = '#ff7f0e'
CLR_GREEN  = '#2ca02c'
CLR_RED    = '#d62728'
CLR_GREY   = '0.5'

# The paper's figures use the modern perceptual colour maps; matplotlib 1.x has
# none of them, so viridis is rebuilt here from its own anchors - identical
# output on any version.
VIRIDIS_ANCHORS = [
    (0.2670, 0.0049, 0.3294), (0.2823, 0.0950, 0.4173), (0.2788, 0.1755, 0.4834),
    (0.2590, 0.2515, 0.5247), (0.2297, 0.3224, 0.5457), (0.1994, 0.3876, 0.5546),
    (0.1727, 0.4488, 0.5579), (0.1490, 0.5081, 0.5573), (0.1276, 0.5669, 0.5506),
    (0.1206, 0.6258, 0.5335), (0.1579, 0.6838, 0.5017), (0.2461, 0.7389, 0.4520),
    (0.3692, 0.7889, 0.3829), (0.5160, 0.8312, 0.2943), (0.6785, 0.8637, 0.1895),
    (0.8456, 0.8873, 0.0997), (0.9932, 0.9062, 0.1439)]
CMAP_SEQ = matplotlib.colors.LinearSegmentedColormap.from_list(
    'viridis_local', VIRIDIS_ANCHORS, N=256)
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    'coolwarm_local',
    [(0.230, 0.299, 0.754), (0.553, 0.690, 0.996), (0.866, 0.866, 0.866),
     (0.968, 0.718, 0.612), (0.706, 0.016, 0.150)], N=256)
CMAP_SEQ.set_bad('black')
CMAP_DIV.set_bad('black')

# One place for the look of every figure. Setting it here means no cell needs to
# repeat grid, font and layout calls.
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.autolayout': True,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'lines.markersize': 4,
})

try:                       # inline figures when running under Jupyter...
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:          # ...and a file-only backend when running as a script
    matplotlib.use('Agg')

print('numpy %s, matplotlib %s' % (np.__version__, matplotlib.__version__))
if int(matplotlib.__version__.split('.')[0]) < 2:
    print('  matplotlib 1.x detected - using explicit hex colours (see above)')


---
# Standalone acquisition register interface

**Purpose.** defines the existing acquisition register map locally. **Hardware.** definitions only. **Wiring.** unchanged. **Pass.** no output. **Failure.** syntax/version problem. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  ACQUISITION SYSTEM - base 0x40600000
# =============================================================================
#  VOLTMETER: tune the local oscillator to the drive, read PEEK_I/PEEK_Q.
#  RECORDER : stream to memory for spectra.
#  ORDER MATTERS: disable -> write settings -> reset -> enable. The decimator
#  holds a running total that assumes a fixed R. Retuning the oscillator is one
#  register write and is safe at any time.
# =============================================================================
class Acq(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = None

    # ---- setup ----
    def start(self, dec_log2, src=0):
        """Bring the chain up at a given decimation. Safe to call repeatedly."""
        self.w(0x00, 0)                 # disable while settings change
        self.w(0x08, dec_log2)          # decimation R = 2**dec_log2
        self.w(0x0C, src)               # 0 = IN1, 1 = IN2
        self.w(0x00, 1 | 2)             # enable + soft reset (self-clearing)
        self.w(0x00, 1)                 # running
        self.dec = dec_log2

    def stop(self):
        self.w(0x00, 0)

    # ---- tuning ----
    def tune_word(self, finc):
        """Set the local oscillator by its raw 32-bit frequency word."""
        self.w(0x04, int(finc) & 0xFFFFFFFF)

    def tune(self, f):
        """Set the local oscillator to the nearest word to f hertz."""
        self.tune_word(int(round(f / FS_CLK * (1 << 32))))

    # ---- voltmeter ----
    def measure(self, navg=None):
        """Average navg in-phase/quadrature pairs; return one complex number.

        At DC both are steady, so this is coherent averaging: a real signal adds
        linearly while noise averages down as the square root of navg."""
        navg = N_AVG if navg is None else navg
        zi = 0.0
        zq = 0.0
        for _ in range(navg):
            zi += self.rs(0x18)
            zq += self.rs(0x1C)
        return complex(zi / navg, zq / navg)

    # ---- housekeeping ----
    def f_out(self):
        """Output sample rate at the current decimation [Hz]."""
        return FS_CLK / (1 << self.dec)

    def window_hz(self):
        """Half-width of the CLEAN part of the receiver window [Hz].

        The decimator rejects by better than 75 dB only over the middle fifth of
        its output band, so the usable half-width is 0.1 x the output rate."""
        return 0.10 * self.f_out()

    def build_id(self):
        return self.r(0x48)

    def status2(self):
        return self.r(0x38)




---
# Standalone generator interface

**Purpose.** defines the existing OUT1 generator interface used only for disconnected dummy-load commissioning and emergency shutdown. **Hardware.** definitions only. **Wiring.** no generated tone may be connected while the resonator loop is assembled. **Pass.** no output. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  SIGNAL GENERATOR - base 0x40200000, stock Red Pitaya module
# =============================================================================
#  Replays a 16384-entry table; frequency is one register write.
#      f = step x 125e6 / 2**30      (about 0.116 Hz per step)
#  Registers: 0x04 amplitude (0x2000 = x1.0), 0x08 table size, 0x0C start,
#  0x10 step, table at 0x10000. Control 0x00: 0x11 free-run, 0xC0 held at zero.
# =============================================================================
class Generator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
        self._step = None

    def load_sine(self):
        """Write one cycle of a sine into the table. Takes a second or two:
        16384 single-word writes, and single-word is not negotiable here."""
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)     # table size, fixed point, inclusive
        self.w(0x0C, 0)                     # start replaying from entry 0

    def freq_word(self, f):
        """Nearest 30-bit frequency word to f hertz."""
        return int(round(f / FS_CLK * (1 << 30)))

    def set_word(self, step):
        """Set frequency by raw word; return the frequency actually produced."""
        self._step = int(step)
        self.w(0x10, self._step)
        return self._step * FS_CLK / float(1 << 30)

    def freq(self, f):
        """Set frequency in hertz; return the frequency actually produced."""
        return self.set_word(self.freq_word(f))

    def lo_word(self):
        """The 32-bit local-oscillator word that lands this signal EXACTLY at
        0 Hz. The generator counts in units of 2**-30 of the clock and the local
        oscillator in units of 2**-32, so the conversion is a factor of four and
        carries no rounding error at all."""
        return 4 * self._step

    def amp(self, frac):
        """Output amplitude as a fraction of full scale. CLAMPED, deliberately.

        The register is Q1.13 with 0x2000 = x1.0, and it happily accepts up to
        0x3FFF = x1.9999. The write below is masked with & 0x3FFF, so before this
        guard existed the mask turned mistakes into silent, WRONG drive levels:

            frac = -0.5  ->  0x3000  ->  x1.5   (a sign slip drives 50% HARDER)
            frac =  2.0  ->  0x0000  ->  x0.0   (silence, and no complaint)
            frac =  1.5  ->  0x3000  ->  x1.5   (clipped square wave, not a sine)

        Every one of those is silent on the bench. Anything outside
        [0, DRIVE_MAX] is now refused instead, so the only way to drive the coil
        harder is to raise DRIVE_MAX in the configuration cell on purpose."""
        frac = float(frac)
        if not np.isfinite(frac):
            raise ValueError('drive amplitude is not a finite number: %r' % frac)
        if frac < 0.0:
            raise ValueError(
                'negative drive amplitude (%.4f). A sign has no meaning here, and '
                'the register would silently wrap it to x%.4f.'
                % (frac, (int(round(frac * 8192)) & 0x3FFF) / 8192.0))
        if frac > DRIVE_MAX:
            raise ValueError(
                'drive %.4f exceeds DRIVE_MAX = %.4f. Nothing in this notebook '
                'may drive harder than the ceiling; raise it in the configuration '
                'cell if that is really what you want.' % (frac, DRIVE_MAX))
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)

    def _control_a_b_zero(self, a_control):
        """Write channel-A control while forcing generator channel B to zero.

        OUT2 is the phase-shifter monitor plus ASG channel B. A plain channel-A
        write clears B's control bits, so stale B state from another program could
        corrupt an external measurement. Bits 23:22 hold B at zero/reset."""
        self.w(0x24, 0)                    # B amplitude and DC offset both zero
        self.w(0x00, 0x00C00000 | int(a_control))

    def assert_b_zero(self):
        """Require channel B amplitude/offset zero and its engine held reset."""
        packed, control = self.r(0x24), self.r(0x00)
        if packed != 0 or ((control >> 22) & 3) != 3:
            raise RuntimeError('generator B is not zero: AMP/DC=0x%08X CTRL=0x%08X'
                               % (packed, control))
        return {'b_amp_dc': packed, 'control': control}

    def run(self):
        self._control_a_b_zero(0x40)       # reset the A read pointer
        self._control_a_b_zero(0x11)       # A free-runs; B remains forced zero

    def off(self):
        self.amp(0.0)
        self._control_a_b_zero(0xC0)       # both outputs held at zero/reset




---
# Standalone OUT1 feedback phase shifter

**Purpose.** defines coefficient, delay, and readback operations for OUT1 feedback. OUT2 is not configured. **Hardware.** definitions only. **Pass.** no output. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  PHASE SHIFTER - base 0x40300000   (Step 8 only)
# =============================================================================
#  Output = A cos(phi) x I + A sin(phi) x Q, Q being the input delayed a quarter
#  period. Two corrections, both from the module header:
#   1. The quarter-period delay is under one clock at 40 MHz, so add whole signal
#      periods and pick the k whose fraction sits nearest 0 or 1.
#   2. Fixed THREE-sample latency (L = 3 since FIX 6; one sample is 115 deg at
#      40 MHz, so a stale value rotates the whole axis). Subtracted from the
#      request so the delivered phase is the requested one.
#  Both describe the DIGITAL path only. Step 8 finds its phases by measurement.
#  "Feedback off" is coefficients (0, 0), NOT the bypass bit.
# =============================================================================
class PhaseShift(Bus):

    LATENCY = 3      # datapath latency in samples (L = 3 since FIX 6; see above)

    def __init__(self):
        Bus.__init__(self, 0x40300000)
        self.f_ref = None

    def config_delay(self, f):
        """Programme the quarter-period delay line for frequency f."""
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000.0:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)       # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        if best is None:
            raise ValueError('no usable quarter-period delay at %.3f MHz' % (f / 1e6))
        self.w(0x08, best[1])                       # whole clocks
        self.w(0x0C, int(round(best[2] * 16384)))   # fraction, Q0.14
        self.w(0x10, 0)                             # bypass OFF
        self.f_ref = f
        return best[1], best[2]

    def set(self, A, phi_deg):
        """Apply gain/phase and prove the exact CI/CQ words were accepted."""
        if self.f_ref is None:
            raise RuntimeError('call config_delay() first')
        if not (0.0 <= A < 1.99):
            # A x 16384 must stay inside int16. At A = 2.0 the coefficient is
            # 32768, which wraps to -32768 and silently INVERTS the feedback.
            raise ValueError('gain must be in [0, 1.99)')
        phi = phi_deg - 360.0 * self.LATENCY * self.f_ref / FS_CLK
        ci = int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF
        cq = int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF
        self.w(0x00, ci); self.w(0x04, cq)
        if self.readback()['ci'] != ci or self.readback()['cq'] != cq:
            raise RuntimeError('phase/gain coefficient readback failed')
        return ci, cq

    def readback(self):
        """All state that can alter feedback or the OUT2 observation."""
        return {'ci': self.r(0x00) & 0xFFFF, 'cq': self.r(0x04) & 0xFFFF,
                'delay': self.r(0x08), 'delta': self.r(0x0C),
                'bypass': self.r(0x10) & 1, 'monitor': self.r(0x18) & 0x3F}

    def assert_held(self, expected):
        got = self.readback()
        changed = {k: (expected[k], got[k]) for k in expected if got[k] != expected[k]}
        if changed:
            raise RuntimeError('feedback/OUT2 settings changed during replay: %r' % changed)
        return got

    def off(self):
        self.w(0x00, 0)
        self.w(0x04, 0)

    def monitor(self, source='input', shift=0):
        """Route a signal to OUT2 and verify the register accepted it.

        source='input' observes the coil/ADC signal used by feedback; 'output'
        observes the phase-shifted feedback contribution. shift is a saturating
        left shift (0..7). Keep it at zero until the external level is known."""
        sources = {'off': 0, 'input': 1, 'output': 2}
        if source not in sources:
            raise ValueError("monitor source must be 'off', 'input' or 'output'")
        shift = int(shift)
        if not (0 <= shift <= 7):
            raise ValueError('monitor shift must be in [0, 7]')
        value = sources[source] | (shift << 3)
        self.w(0x18, value)
        got = self.r(0x18)
        if (got & 0x3F) != value:
            raise RuntimeError('OUT2 monitor readback failed: wrote 0x%02X, read 0x%08X; check the bitstream' % (value, got))
        return got

    def monitor_clipped(self):
        """True if OUT2 monitor scaling has clipped since configuration."""
        return bool(self.r(0x18) & (1 << 8))

    def monitor_off(self):
        self.monitor('off', 0)



---
# Standalone raw scope interface

**Purpose.** defines ADC headroom and clipping captures. **Hardware.** definitions only. **Wiring.** IN1 remains directly connected in native high-impedance mode; no external terminator is fitted. **Pass.** no output. **Safe to rerun.** yes.


In [ ]:
# =============================================================================
#  INTERNAL SCOPE - base 0x40100000
# =============================================================================
#  16384 raw samples at up to 125 MSa/s. The only overload guard on the real
#  path: the emulator's sticky clip flag is gone once a real coil is measured.
#  Sees the same signal as the rest of the chain, so with the emulator enabled it
#  shows the emulator.
# =============================================================================
SC_BASE = 0x40100000
SC_SPAN = 0x40000
SC_BUF_A = 0x10000
SC_BUF_B = 0x20000
SC_NMAX = 1 << 14


class Scope(Bus):

    def __init__(self):
        Bus.__init__(self, SC_BASE, span=SC_SPAN)

    def _buf(self, off, n):
        """Read n samples, one 32-bit word at a time (this is a register block,
        not DDR). The buffer holds 14-bit two's-complement values."""
        ba = bytearray(4 * n)
        m = self.m
        for i in range(n):
            o = off + 4 * i
            ba[4 * i:4 * i + 4] = m[o:o + 4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=SC_NMAX, chan='A'):
        """Capture and return (samples, sample_rate). dec = 1 is every sample."""
        self.w(0x14, dec)
        self.w(0x10, n - 1)
        self.w(0x00, 2)        # reset the capture engine
        self.w(0x00, 1)        # arm
        self.w(0x04, 1)        # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        off = SC_BUF_A if chan == 'A' else SC_BUF_B
        return self._buf(off, n), FS_CLK / float(dec)

    def peak_frac(self, n=4096):
        """TOTAL peak excursion as a fraction of full scale. 1.0 means clipping.

        Deliberately includes everything - the converter's DC offset, broadband
        noise, out-of-band pickup - because all of it counts towards clipping.
        That makes it the right guard and the WRONG way to measure the size of a
        tone. Use tone_frac() for that."""
        x, _ = self.acquire(dec=1, n=n)
        return float(np.max(np.abs(x))) / 8191.0

    def tone_frac(self, f, n=4096):
        """Amplitude of the component at f alone, as a fraction of full scale.

        WHY THIS EXISTS. peak_frac() answers "how close am I to clipping". It was
        also being used to answer "how much 40 MHz is arriving", and those are
        not the same question. max|x| is inflated by the converter's DC offset
        and by every other thing in the record, so on this board it read about
        four times the true tone amplitude - which then over-predicted the
        feedback loop gain by the same factor.

        A least-squares projection onto cos and sin at the known frequency fixes
        it: the mean is removed first, so DC cannot contribute, and anything not
        at f is orthogonal to both basis vectors and averages away. No window and
        no scalloping loss, because f is known exactly rather than searched."""
        x, fs = self.acquire(dec=1, n=n)
        x = x - np.mean(x)                       # the converter's DC offset
        t = np.arange(len(x)) / fs
        basis = np.column_stack([np.cos(2.0 * np.pi * f * t),
                                 np.sin(2.0 * np.pi * f * t)])
        coef, _, _, _ = np.linalg.lstsq(basis, x, rcond=-1)
        return float(np.hypot(coef[0], coef[1])) / 8191.0



---
# Standalone DMA capture interface

**Purpose.** defines bounded complex-IQ capture. Real DMA memory is discovered lazily; tests inject a capture backend. **Hardware.** no access until capture is called. **Pass.** definitions complete. **Safe to rerun.** yes before acquisition.


In [ ]:
# =============================================================================
#  MEMORY CAPTURE
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None


MEM_START = None
MEM_SIZE = 0x00800000
MEM_STOP = None
POISON = 0xDEADBEEF


def configure_capture_memory():
    """Discover safe DMA bounds only when real hardware capture is requested."""
    global MEM_START, MEM_STOP
    blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
    if not blob or len(blob) < 8:
        raise RuntimeError('cannot read the device-tree memory node')
    declared_end = _be32(blob, 0) + _be32(blob, 1)
    if declared_end >= 512 * 1024 ** 2:
        raise RuntimeError('no safe memory above the declared end')
    MEM_START = (declared_end + 0xFFF) & ~0xFFF
    MEM_STOP = MEM_START + (MEM_SIZE & ~7) - 8
    print('capture region 0x%08X .. 0x%08X inclusive (%d MB)' %
          (MEM_START, MEM_STOP + 7, MEM_SIZE // (1024 * 1024)))

def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((nbytes + 4095) // 4096) * 4096, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)


def _read_region(nbytes):
    """Bulk read of DDR. try/finally because everything here can raise, and a
    leaked mapping inside a scan loop surfaces much later as an unrelated-
    looking out-of-memory error."""
    fd = mm = None
    try:
        fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
        return mm[0:nbytes]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)


def capture(lo_word, dec_log2, n_samples, settle=None):
    """Record n_samples of the complex stream. Returns a complex numpy array.

    lo_word is the RAW 32-bit local-oscillator word, so a caller that wants an
    exact zero-frequency landing can pass gen.lo_word() and get it. Raises on
    any transport fault rather than returning plausible garbage."""
    settle = SETTLE if settle is None else settle
    if CAPTURE_BACKEND is not None:
        return CAPTURE_BACKEND(lo_word, dec_log2, n_samples, settle)
    if MEM_START is None:
        configure_capture_memory()
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples exceeds the capture region' % n_samples)

    _poison(n_samples * 8)

    acq.w(0x00, 0)                       # disable while settings change
    acq.w(0x08, dec_log2)
    acq.w(0x0C, 0)
    acq.dec = dec_log2
    acq.tune_word(lo_word)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)

    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded memory writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))

    acq.w(0x00, 1 | 2)                   # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                   # ...let the pipeline flush...
    acq.w(0x00, 1 | 16 | 128)            # ...then arm with a fresh writer
    acq.w(0x00, 1 | 16)

    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                       # disarm
    time.sleep(0.002)                    # let the last burst land

    st2 = acq.r(0x38)
    if st2 & 8:
        raise RuntimeError('memory refused writes - run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    if got < n_samples:
        raise RuntimeError('short record: asked %d, got %d' % (n_samples, got))

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    limit = max(8, 0.001 * u32.size)
    if int(np.count_nonzero(u32 == POISON)) > limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms.
        time.sleep(0.02)
        u32 = np.frombuffer(_read_region(got * 8), dtype='<u4')
        if int(np.count_nonzero(u32 == POISON)) > limit:
            raise RuntimeError('unwritten words in the record - transport fault')

    iq = u32.astype(np.int64)
    i = np.where(iq[0::2] & 0x80000000, iq[0::2] - (1 << 32), iq[0::2])
    q = np.where(iq[1::2] & 0x80000000, iq[1::2] - (1 << 32), iq[1::2])
    return i.astype(float) + 1j * q.astype(float)


---
# 3 — Hardware identity and two shutdown levels

**Purpose.** opens the existing register blocks, performs a full shutdown, and verifies the FPGA identity. OUT2 is initially off. `feedback_off()` later preserves a commissioned campaign monitor; `full_shutdown()` always disables it.

**Wiring.** Keep the declared direct high-impedance IN1 connection unchanged. Keep OUT1 disconnected from the air-gap pin for commissioning. **Expected result.** zero generator and feedback coefficients, OUT2 off, acquisition stopped. **Safe to rerun.** yes.


In [ ]:
acq = Acq()
gen = Generator()
ps = PhaseShift()
sc = Scope()
OUT2_MONITOR_CALIBRATION = None
OUT1_DC_COMMISSIONING = {'accepted': True,
    'method': 'digital_zero_state_plus_open_airgap_assumption',
    'analogue_offset_directly_measured': False,
    'airgap_open': True, 'return_attenuation_db_nominal': 30.0}

def feedback_off():
    """Clear OUT1 feedback and both generators while preserving a healthy OUT2 monitor."""
    errors = []
    for label, action in (('generator', gen.off), ('feedback', ps.off), ('acquisition', acq.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        readback = ps.readback()
        if readback['ci'] or readback['cq']:
            errors.append('feedback coefficient readback is nonzero')
        gen.assert_b_zero()
    except BaseException as exc:
        errors.append('readback: %s' % exc)
    if errors:
        try:
            ps.monitor_off()
        finally:
            raise RuntimeError('feedback shutdown failed; disconnect OUT1: ' + '; '.join(errors))
    print('SAFE FEEDBACK STATE: generators and OUT1 zero; OUT2 monitor state preserved.')
    return True

def out2_monitor_off():
    """Disable only the IN1-to-OUT2 monitor and verify its register readback."""
    ps.monitor_off()
    state = ps.readback()
    if state['monitor']:
        full_shutdown()
        raise RuntimeError('OUT2 monitor disable readback failed; full shutdown attempted')
    print('OUT2 monitor disabled and verified. OUT1 feedback state was not changed.')
    return True

def full_shutdown():
    """Attempt every safe action, including disabling the OUT2 connector."""
    errors = []
    for label, action in (('generator', gen.off), ('feedback', ps.off),
                          ('OUT2 monitor', ps.monitor_off), ('acquisition', acq.stop)):
        try:
            action()
        except BaseException as exc:
            errors.append('%s: %s' % (label, exc))
    try:
        state = ps.readback()
        gen.assert_b_zero()
        if state['ci'] or state['cq'] or state['monitor']:
            errors.append('nonzero final register state: %r' % state)
    except BaseException as exc:
        errors.append('final readback: %s' % exc)
    if errors:
        raise RuntimeError('FULL SHUTDOWN COULD NOT BE VERIFIED: ' + '; '.join(errors))
    print('FULL SAFE STATE: generators, OUT1, OUT2 and acquisition are off.')
    return True

def disable_everything():
    """Memorable emergency alias: disable generators, feedback, OUT2 and acquisition."""
    return full_shutdown()

full_shutdown()
CURRENT_BUILD_ID = '0x%08X' % acq.build_id()
if CURRENT_BUILD_ID.lower() != str(BUILD_ID_EXPECTED).lower():
    full_shutdown()
    raise RuntimeError('wrong FPGA identity: found %s, expected %s' %
                       (CURRENT_BUILD_ID, BUILD_ID_EXPECTED))
print('PASS: qualified FPGA identity', CURRENT_BUILD_ID)


---
# 4 — Measure IN1 safety and identify the source of any DC offset

**What this cell does.** It acquires repeated raw IN1 records with feedback off and reports the
signed mean voltage as well as the offset-removed RF/noise waveform. A negative mean is not a
negative RF amplitude: it means the ADC codes sit below the converter's measured zero.

**Why it is needed.** The 2.2 pF sensing capacitor blocks coil-side DC, but it does not remove ADC
offset or DC generated downstream by the follower/ZFL. Measurements with bare high-impedance IN1, the
powered reference chain, and the complete resonator separate these contributions.

**Hardware.** OUT1 feedback and both generators are forced off. Power the ZFL down before changing
its output connection. This bench intentionally presents native high-impedance IN1 rather than a 50 ohm load, so verify the ZFL remains stable, does not oscillate and stays within its output limits before feedback commissioning.

**Pass criteria.** Total peak remains below 0.80 V and |DC| below 0.10 V. |DC| above 0.05 V is a
warning requiring source isolation. Raw DC always counts towards headroom; subtraction is used only
for plots and spectral analysis, never as a hidden hardware correction. Safe to rerun with feedback off.


In [ ]:
INPUT_DC_OFFSETS = {}


def measure_live_input_safety(label='complete_resonator_feedback_off',
                              samples=16384, repeats=5):
    """Measure total headroom and stable signed ADC offset with feedback off."""
    feedback_off()
    records = []
    sample_rate = None
    for repeat in range(int(repeats)):
        raw, sample_rate = sc.acquire(dec=1, n=samples, chan='A')
        records.append(raw * ADC_RAW_VOLTS_PER_COUNT)
    values = np.asarray(records, dtype=float)
    means = np.mean(values, axis=1)
    dc = float(np.mean(means))
    dc_uncertainty = float(np.std(means, ddof=1)/np.sqrt(len(means)))
    ac = values-values.mean(axis=1, keepdims=True)
    peak = float(np.max(np.abs(values)))       # includes DC: correct clipping guard
    ac_peak = float(np.max(np.abs(ac)))
    rms = float(np.sqrt(np.mean(ac**2)))
    dc_stability = float(np.std(means, ddof=1))
    warnings = []
    if abs(dc) >= IN1_DC_OFFSET_WARNING_V:
        warnings.append(
            '|mean DC| %.6g V exceeds the %.6g V warning level; compare bare high-impedance '
            'IN1 and powered-chain conditions before feedback' %
            (abs(dc), IN1_DC_OFFSET_WARNING_V))
    accepted = bool(peak < ADC_OPERATING_GUARD_VPK and
                    abs(dc) < IN1_DC_OFFSET_MAX_V)
    result = {'label': label, 'accepted': accepted, 'peak_v_including_dc': peak,
              'ac_peak_v': ac_peak, 'ac_rms_v': rms, 'dc_offset_v': dc,
              'dc_offset_uncertainty_v': dc_uncertainty,
              'dc_record_to_record_sd_v': dc_stability,
              'mean_adc_code': dc/ADC_RAW_VOLTS_PER_COUNT,
              'sample_rate_hz': sample_rate, 'repeats': int(repeats),
              'headroom_v': ADC_OPERATING_GUARD_VPK-peak,
              'dc_warning_level_v': IN1_DC_OFFSET_WARNING_V,
              'dc_rejection_level_v': IN1_DC_OFFSET_MAX_V,
              'warnings': warnings,
              'interpretation': ('signed offset relative to ADC zero; not a negative RF amplitude')}
    INPUT_DC_OFFSETS[label] = result
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
    time_us = np.arange(samples)/sample_rate*1e6
    axes[0].plot(time_us, values[0], color=CLR_BLUE, linewidth=0.7)
    axes[0].axhline(dc, color=CLR_ORANGE, label='mean %.3f mV' % (dc*1e3))
    axes[0].set(xlabel='Time (microseconds)', ylabel='Raw IN1 voltage (V)',
                title='Raw waveform: DC remains in headroom')
    axes[0].legend()
    axes[1].plot(time_us, ac[0], color=CLR_GREEN, linewidth=0.7)
    axes[1].set(xlabel='Time (microseconds)', ylabel='Offset-removed voltage (V)',
                title='RF/noise view after mean subtraction')
    axes[2].hist(values.ravel(), bins=80, color=CLR_BLUE, alpha=0.8)
    axes[2].axvline(dc, color=CLR_ORANGE)
    axes[2].set(xlabel='Raw IN1 voltage (V)', ylabel='Samples',
                title='Distribution and signed offset')
    plt.tight_layout(); plt.show()
    print(json.dumps(result, indent=2))
    feedback_off()
    if not accepted:
        raise RuntimeError(
            'IN1 safety rejected condition %r: total peak %.6g V (guard %.6g V), '
            'mean DC %.6g ± %.3g V (limit %.6g V). Feedback remains off. Measure '
            'IN1, then the powered chain, to identify the source.' %
            (label, peak, ADC_OPERATING_GUARD_VPK, dc, dc_uncertainty,
             IN1_DC_OFFSET_MAX_V))
    return result


def compare_input_dc_offsets(offset_records=INPUT_DC_OFFSETS):
    """Separate native-IN1 offset from follower/ZFL and resonator changes."""
    labels = ('in1_high_impedance_zfl_disconnected', 'powered_detection_chain',
              'complete_resonator_feedback_off')
    missing = [label for label in labels if label not in offset_records]
    if missing:
        raise CalibrationError(
            'DC-source comparison is missing: %s. Measure the three conditions in '
            'order with feedback off; do not infer amplifier DC from one capture.' %
            ', '.join(missing))
    adc = float(offset_records[labels[0]]['dc_offset_v'])
    powered = float(offset_records[labels[1]]['dc_offset_v'])
    connected = float(offset_records[labels[2]]['dc_offset_v'])
    result = {'adc_native_input_offset_v': adc,
              'powered_chain_increment_v': powered-adc,
              'resonator_connection_increment_v': connected-powered,
              'complete_measured_offset_v': connected,
              'accepted': bool(all(row['accepted'] for row in offset_records.values()) and
                               abs(connected) < IN1_DC_OFFSET_MAX_V)}
    print(json.dumps(result, indent=2))
    return result


LIVE_INPUT_SAFETY = measure_live_input_safety()

# To identify the source of a persistent negative mean, safely measure all three:
# adc_dc = measure_live_input_safety('in1_high_impedance_zfl_disconnected')
# chain_dc = measure_live_input_safety('powered_detection_chain')
# resonator_dc = measure_live_input_safety('complete_resonator_feedback_off')
# dc_source_report = compare_input_dc_offsets()


---
# 5 — Optional disconnected electronics self-test

**Purpose.** Verify generator, receiver, scaling and shutdown against a known dummy path before
trusting an unknown resonator.

**Hardware.** Only when explicitly enabled, with OUT1 and IN1 disconnected from the resonator and
joined through a rated 50 ohm calibration path. **Expected result.** Known frequency and amplitude
without clipping. **Pass.** Scaling and shutdown readback agree with the dummy path. **Failure.**
Repair the instrument path before feedback. **Safe to rerun.** Yes, on the dummy path only.


In [ ]:
def run_disconnected_selftest(frequency_hz, drive_fraction=0.01):
    if not 0 < drive_fraction <= 0.05:
        raise ValueError('Dummy-load drive_fraction must be greater than 0 and no more than 0.05; received %.9g. Disconnect the resonator, connect the rated 50 ohm dummy path, and choose a smaller positive fraction.' % float(drive_fraction))
    try:
        ps.off()
        gen.load_sine()
        gen.freq(frequency_hz)
        gen.amp(drive_fraction)
        gen.run()
        time.sleep(SETTLE)
        peak = sc.peak_frac(4096)
        print('Dummy-load peak fraction of ADC full scale:', peak)
        return {'frequency_hz': frequency_hz, 'drive_fraction': drive_fraction,
                'adc_peak_fraction': peak}
    finally:
        gen.off()
        ps.off()
# Do not call this function while connected to the resonator or air-gap pin.


---
# 6 — Passive acquisition and capture-health functions

**What this cell does.** It defines supported complex-baseband capture, ADC scaling, PSD
normalisation, CIC-response prediction, optional measured transfer correction, and record-health
checks. Merely running the cell does not acquire data.

**Why it is needed.** Mixing 39 MHz to baseband makes narrow measurements practical, but the CIC
decimator attenuates frequencies away from the LO. Below R=256 the 309-tap FIR cannot keep up and
is automatically bypassed. The notebook must therefore preserve both raw and corrected spectra
and must never treat the full FFT interval as flat.

**Hardware and wiring.** None until `acquire_passive_records()` is called. When called, IN1 must
remain connected directly in its native high-impedance mode; no input pad or terminator is fitted. OUT1 coefficients and both
generators are forced to zero for passive captures.

**Expected output.** Function definitions only. Later cells plot raw PSD, estimated/measured
acquisition gain, corrected PSD and rejected regions. Capture health reports peak voltage,
headroom, crest factor, clipping and record-to-record stationarity.

**Pass/failure/rerun.** Parseval, finite-data and headroom checks must pass. Unsupported
decimation, excessive response correction, zero/invalid records or nonstationarity are rejected
with the measured values and corrective action. Safe to rerun after feedback is off.


In [ ]:
def _lo_word(frequency_hz):
    return int(round(float(frequency_hz) / FS_CLK * (1 << 32))) & 0xFFFFFFFF


def cic_power_response(frequency_hz, plan, stages=4):
    """Normalized CIC PSD gain versus absolute input frequency.

    This large-R sinc approximation is diagnostic. A swept-tone calibration is
    preferred for accepted amplitude work because it also includes mixer, FIR,
    ADC, cabling and analogue-chain response.
    """
    frequency = np.asarray(frequency_hz, dtype=float)
    relative = (frequency - float(plan['centre_hz'])) / float(plan['sample_rate_hz'])
    return np.maximum(np.abs(np.sinc(relative)) ** (2 * int(stages)),
                      np.finfo(float).tiny)


def acquisition_response_power(frequency_hz, plan, calibration=None):
    """Return PSD gain and provenance for correction and plotting."""
    frequency = np.asarray(frequency_hz, dtype=float)
    if calibration is not None:
        measured_f = np.asarray(calibration.get(
            'offset_hz', np.asarray(calibration['frequency_hz'], dtype=float) -
            float(calibration.get('lo_centre_hz', plan['centre_hz']))), dtype=float)
        measured_p = np.asarray(calibration['power_gain'], dtype=float)
        requested_offset = frequency - float(plan['centre_hz'])
        gain = np.interp(requested_offset, measured_f, measured_p, left=np.nan, right=np.nan)
        return gain, 'measured swept-tone response'
    if int(plan['decimation_log2']) < FIR_MIN_DECIMATION_LOG2:
        return cic_power_response(frequency, plan), 'modelled CIC response; FIR bypassed'
    return np.ones_like(frequency), 'nominal corrected response; swept-tone verification required'


def correct_acquisition_response(spectrum, plan, calibration=None,
                                 maximum_correction_db=MAX_RESPONSE_CORRECTION_DB):
    """Correct PSD only where response gain and correction remain trustworthy."""
    frequency = np.asarray(spectrum['frequency_hz'], dtype=float)
    raw = np.asarray(spectrum['psd_v2_per_hz'], dtype=float)
    gain, source = acquisition_response_power(frequency, plan, calibration)
    minimum_gain = 10.0 ** (-float(maximum_correction_db) / 10.0)
    valid = np.isfinite(gain) & (gain >= minimum_gain)
    corrected = np.full(raw.shape, np.nan, dtype=float)
    corrected[valid] = raw[valid] / gain[valid]
    out = dict(spectrum)
    out.update({'raw_psd_v2_per_hz': raw, 'psd_v2_per_hz': corrected,
                'acquisition_power_gain': gain,
                'response_correction_valid': valid,
                'response_source': source,
                'maximum_response_correction_db': float(maximum_correction_db)})
    return out

def acquisition_plan(f0_hz, q_estimate, expected_shift_hz=0.0, centre_on_shift=False):
    plan = plan_passive_acquisition(
        f0_hz, q_estimate, expected_shift_hz, clock_hz=FS_CLK,
        min_decimation_log2=MIN_ACQUISITION_DECIMATION_LOG2,
        max_decimation_log2=MAX_ACQUISITION_DECIMATION_LOG2,
        bins_per_linewidth=BINS_PER_LINEWIDTH,
        linewidths_each_side=LINEWIDTHS_EACH_SIDE,
        max_samples=MAX_CAPTURE_SAMPLES,
        centre_on_expected_shift=centre_on_shift)
    print('Adaptive plan: span %.1f kHz, bin %.3f Hz, %.1f bins/linewidth, %.2f s/record' %
          (plan['full_span_hz'] / 1e3, plan['bin_hz'],
           plan['bins_per_linewidth'], plan['record_duration_s']))
    return plan

def acquire_passive_records(plan=None, averages=PASSIVE_AVERAGES):
    """Acquire natural resonator noise; never enable a generated tone."""
    plan = INITIAL_ACQUISITION_PLAN if plan is None else plan
    gen.off()
    records = []
    try:
        for number in range(int(averages)):
            raw = capture(_lo_word(plan['centre_hz']), plan['decimation_log2'],
                          plan['samples_per_record'], settle=SETTLE)
            records.append(raw * float(CIC_VOLTS_PER_COUNT))
            if number in (0, int(averages) - 1):
                print('captured passive record %d/%d' % (number + 1, averages))
    except BaseException:
        feedback_off()
        raise
    return np.asarray(records)

def make_passive_spectrum(records, plan=None):
    plan = INITIAL_ACQUISITION_PLAN if plan is None else plan
    return passive_iq_spectrum(records, plan['sample_rate_hz'], plan['centre_hz'])

def record_health(records, stationarity_cv_limit=0.20):
    """Assess finite values, converter margin, crest factor and record stationarity."""
    values = np.asarray(records)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 16:
        raise CalibrationError(
            'Passive record health requires a 2D array with at least two records '
            'and 16 samples per record; received shape %r.' % (values.shape,))
    if not np.all(np.isfinite(values)):
        bad = int(np.size(values) - np.count_nonzero(np.isfinite(values)))
        raise CalibrationError(
            'Passive records contain %d non-finite samples. Discard the capture; '
            'check DMA completion, scaling and buffer bounds before retrying.' % bad)
    magnitude = np.abs(values)
    peak = float(np.max(magnitude))
    rms_each = np.sqrt(np.mean(magnitude ** 2, axis=1))
    mean_rms = float(np.mean(rms_each))
    if mean_rms <= np.finfo(float).tiny:
        raise CalibrationError(
            'Every passive record is numerically zero. Check IN1 cabling, ADC range, '
            'DMA source and voltage scaling; a zero capture cannot establish safety.')
    cv = float(np.std(rms_each, ddof=1) / mean_rms)
    full_scale_vpk = float(RP_LIMITS['input_lv_full_scale_vpk'])
    operating_guard_vpk = float(ADC_OPERATING_GUARD_VPK)
    clipped = bool(peak >= 0.98 * full_scale_vpk)
    within_guard = bool(peak < operating_guard_vpk)
    stationary = bool(cv <= float(stationarity_cv_limit))
    result = {
        'peak_v': peak,
        'headroom_to_operating_guard_v': operating_guard_vpk - peak,
        'headroom_to_full_scale_v': full_scale_vpk - peak,
        'rms_v': mean_rms,
        'crest_factor': peak / mean_rms,
        'rms_cv': cv,
        'stationarity_cv_limit': float(stationarity_cv_limit),
        'clipped': clipped,
        'within_operating_guard': within_guard,
        'stationary': stationary,
    }
    reasons = []
    if clipped:
        reasons.append('peak %.6g V reached 98%% of the %.6g Vpk LV full scale' %
                       (peak, full_scale_vpk))
    elif not within_guard:
        reasons.append('peak %.6g V exceeded the conservative %.6g Vpk operating guard' %
                       (peak, operating_guard_vpk))
    if not stationary:
        reasons.append('record RMS coefficient of variation %.4f exceeded %.4f' %
                       (cv, stationarity_cv_limit))
    result['rejection_reasons'] = reasons
    result['accepted'] = not reasons
    return result


---
# 7 — Calibrate the ADC, FFT and acquisition-filter response

**What this cell does.** It defines independent gates for ADC volts/count, FFT Parseval scaling,
I/Q balance, alias rejection, analyser agreement and a swept-tone measurement of the complete
acquisition response.

**Why it is needed.** A passive noise dome can be produced by the CIC even with a perfectly flat
analogue input. A constant-amplitude tone stepped through 38.5–39.5 MHz distinguishes expected
digital gain from an analogue-chain fault and tests whether the FIR is actually correcting the
response when active.

**Hardware procedure for the swept-tone test.** Keep feedback off and disconnect the OUT1 air-gap
return. Apply a small, independently measured external tone through the established safe input
geometry. Randomise the frequency order, repeat the centre and endpoints, and remain well below
the measured ZFL/ADC compression point. Record input RMS and the corresponding digital RMS. Never
enable a generated tone automatically on the resonator/trap setup.

**Expected plots.** Measured power gain and the predicted CIC curve versus LO offset, plus residual
error. At R=64 or 128 the FIR must be reported as bypassed and CIC droop is expected. At R>=256 the
FIR should be active and the central response should agree with its generated design.

**Pass criteria.** At least nine finite frequency points spanning the full accepted search band,
repeated-point scatter below 2%, no compression/hysteresis, and no correction exceeding 12 dB.
Failure means the response cannot yet be removed reliably; passive Q fitting remains diagnostic.
Function definitions are safe to rerun.


In [ ]:
def calibrate_adc_scale(reference_vrms, measured_count_rms):
    """Calibrate volts/count from repeated traceable RMS measurements."""
    reference = np.asarray(reference_vrms, dtype=float)
    counts = np.asarray(measured_count_rms, dtype=float)
    if reference.shape != counts.shape or reference.size < 3:
        raise CalibrationError('ADC scale needs at least three paired measurements')
    if np.any(reference <= 0) or np.any(counts <= 0):
        raise CalibrationError('ADC calibration values must be positive')
    scales = reference / counts
    relative_scatter = float(np.std(scales, ddof=1) / np.mean(scales))
    return {'accepted': relative_scatter <= 0.02,
            'volts_per_count': float(np.mean(scales)),
            'uncertainty_volts_per_count': float(np.std(scales, ddof=1)),
            'relative_scatter': relative_scatter,
            'coil_voltage_directly_measured': False,
            'airgap_transfer_directly_measured': False}


def validate_fft_parseval(records, spectrum, maximum_fractional_error=0.03):
    """Check that integrated PSD equals time-domain mean-square voltage."""
    values = np.asarray(records)
    time_power = float(np.mean(np.abs(values - np.mean(values, axis=1, keepdims=True)) ** 2))
    frequency_power = float(np.sum(spectrum['psd_v2_per_hz']) * spectrum['bin_hz'])
    error = abs(frequency_power / max(time_power, np.finfo(float).tiny) - 1.0)
    return {'accepted': error <= maximum_fractional_error,
            'time_domain_power_v2': time_power,
            'integrated_psd_power_v2': frequency_power,
            'fractional_error': float(error)}


def assess_iq_quality(records, maximum_gain_imbalance=0.05,
                      maximum_correlation=0.05):
    """Measure I/Q RMS imbalance and normalized cross-correlation."""
    values = np.asarray(records, dtype=complex)
    i = values.real.ravel() - np.mean(values.real)
    q = values.imag.ravel() - np.mean(values.imag)
    irms = float(np.sqrt(np.mean(i ** 2)))
    qrms = float(np.sqrt(np.mean(q ** 2)))
    imbalance = abs(irms / max(qrms, np.finfo(float).tiny) - 1.0)
    correlation = abs(float(np.mean(i * q) /
                      max(irms * qrms, np.finfo(float).tiny)))
    return {'accepted': imbalance <= maximum_gain_imbalance and
                        correlation <= maximum_correlation,
            'i_rms': irms, 'q_rms': qrms,
            'gain_imbalance': float(imbalance),
            'normalized_correlation': float(correlation)}


def assess_alias_rejection(in_band_power, aliased_power,
                           minimum_rejection_db=60.0):
    """Require an injected out-of-band test signal to remain below the limit."""
    rejection = 10.0 * np.log10(max(float(in_band_power), np.finfo(float).tiny) /
                                max(float(aliased_power), np.finfo(float).tiny))
    return {'accepted': rejection >= minimum_rejection_db,
            'rejection_db': float(rejection),
            'minimum_rejection_db': float(minimum_rejection_db)}


def assess_noise_campaign(noise_records, required_labels=None,
                          maximum_within_condition_cv=0.20):
    """Validate three different noise conditions without calling their difference drift.

    The bare high-impedance ADC, powered amplifier and connected resonator are expected
    to have different floors.  Stationarity is evaluated within each condition;
    between-condition ratios diagnose which part of the chain limits sensitivity.
    """
    required = required_labels or ('in1_high_impedance_zfl_disconnected',
                                   'powered_detection_chain',
                                   'complete_resonator_feedback_off')
    missing = [label for label in required if label not in noise_records]
    rejected = {}
    for label in required:
        if label not in noise_records:
            continue
        row = noise_records[label]
        reasons = []
        if not row['health'].get('accepted', False):
            reasons.append('record-health gate failed')
        floor_cv = float(row.get('floor_relative_scatter', float('inf')))
        if not np.isfinite(floor_cv) or floor_cv > maximum_within_condition_cv:
            reasons.append('within-condition floor CV %.4f exceeds %.4f' %
                           (floor_cv, maximum_within_condition_cv))
        if reasons:
            rejected[label] = reasons
    ratios = {}
    if not missing:
        adc = float(noise_records[required[0]]['median_floor_v2_per_hz'])
        chain = float(noise_records[required[1]]['median_floor_v2_per_hz'])
        resonator = float(noise_records[required[2]]['median_floor_v2_per_hz'])
        ratios = {'chain_to_adc_power_ratio': chain/max(adc, np.finfo(float).tiny),
                  'resonator_to_chain_power_ratio': resonator/max(chain, np.finfo(float).tiny),
                  'chain_to_adc_db': 10*np.log10(chain/max(adc, np.finfo(float).tiny)),
                  'resonator_to_chain_db': 10*np.log10(
                      resonator/max(chain, np.finfo(float).tiny))}
    return {'accepted': not missing and not rejected,
            'missing_conditions': missing, 'rejected_conditions': rejected,
            'within_condition_cv_limit': float(maximum_within_condition_cv),
            'diagnostic_floor_ratios': ratios}


def validate_cic_voltage_scale(raw_cic_peak_counts, known_input_vpk,
                               maximum_fractional_error=0.05):
    """Check the RTL-derived 2^29 CIC scale with a small known input tone."""
    counts = float(raw_cic_peak_counts)
    known = float(known_input_vpk)
    if not np.isfinite(counts) or not np.isfinite(known) or counts <= 0 or known <= 0:
        raise CalibrationError(
            'CIC scale validation needs positive finite raw peak counts and known '
            'IN1 peak volts. Use a small independently measured tone with feedback off.')
    reconstructed = counts * CIC_VOLTS_PER_COUNT
    error = abs(reconstructed/known - 1.0)
    return {'accepted': bool(error <= maximum_fractional_error),
            'raw_cic_peak_counts': counts, 'known_input_vpk': known,
            'reconstructed_input_vpk': reconstructed,
            'fractional_error': error,
            'maximum_fractional_error': float(maximum_fractional_error),
            'cic_counts_per_adc_vpk': CIC_COUNTS_PER_ADC_VPK}

def assess_analyser_agreement(comparison, linewidth_hz,
                              centre_tolerance_linewidths=0.20,
                              q_fraction_tolerance=0.10):
    """Check independent centre and Q estimates without asserting amplitude."""
    centre_fraction = abs(float(comparison['f0_difference_hz'])) / max(
        float(linewidth_hz), np.finfo(float).tiny)
    q_fraction = abs(float(comparison['q_fractional_difference']))
    return {'accepted': centre_fraction <= centre_tolerance_linewidths and
                        q_fraction <= q_fraction_tolerance,
            'centre_difference_linewidths': float(centre_fraction),
            'q_fractional_difference': float(q_fraction),
            'amplitude_comparison_is_relative_only': True}


def combine_acquisition_calibration(adc, parseval, iq, alias):
    """Produce the single acquisition gate stored in the sealed calibration."""
    checks = {'adc_scale': adc.get('accepted', False),
              'fft_parseval': parseval.get('accepted', False),
              'iq_quality': iq.get('accepted', False),
              'alias_rejection': alias.get('accepted', False)}
    return {'accepted': all(checks.values()), 'checks': checks,
            'adc': adc, 'parseval': parseval, 'iq': iq, 'alias': alias}



def calibrate_acquisition_response(tone_frequency_hz, input_vrms,
                                   measured_output_rms, plan,
                                   maximum_relative_scatter=0.02):
    """Build an empirical acquisition power-gain calibration from safe tones."""
    frequency = np.asarray(tone_frequency_hz, dtype=float)
    input_level = np.asarray(input_vrms, dtype=float)
    output_level = np.asarray(measured_output_rms, dtype=float)
    if not (frequency.shape == input_level.shape == output_level.shape):
        raise CalibrationError(
            'Acquisition-response arrays must have identical shapes; received '
            'frequency %r, input %r and output %r.' %
            (frequency.shape, input_level.shape, output_level.shape))
    if frequency.size < 9 or np.any(~np.isfinite(frequency + input_level + output_level)):
        raise CalibrationError(
            'Acquisition-response calibration needs at least nine finite tone points '
            'covering 38.5–39.5 MHz, including repeated centre and edge measurements.')
    if np.any(input_level <= 0) or np.any(output_level <= 0):
        raise CalibrationError(
            'Every calibration RMS value must be positive. Zero or negative values '
            'usually indicate a failed capture, wrong units or a noise-limited tone.')
    low = CENTRE_HZ - DISCOVERY_SEARCH_HALF_WIDTH_HZ
    high = CENTRE_HZ + DISCOVERY_SEARCH_HALF_WIDTH_HZ
    if np.min(frequency) > low or np.max(frequency) < high:
        raise CalibrationError(
            'Tone sweep covers %.6f..%.6f MHz but must span at least %.6f..%.6f MHz. '
            'Extend the sweep before using it to correct passive spectra.' %
            (np.min(frequency)/1e6, np.max(frequency)/1e6, low/1e6, high/1e6))
    order = np.argsort(frequency)
    frequency = frequency[order]
    voltage_gain = (output_level / input_level)[order]
    power_gain = voltage_gain ** 2
    centre_gain = float(np.interp(CENTRE_HZ, frequency, power_gain))
    power_gain /= max(centre_gain, np.finfo(float).tiny)
    unique = np.unique(frequency)
    repeated_scatter = []
    for value in unique:
        group = power_gain[frequency == value]
        if group.size > 1:
            repeated_scatter.append(float(np.std(group, ddof=1) /
                                          max(np.mean(group), np.finfo(float).tiny)))
    worst_scatter = max(repeated_scatter) if repeated_scatter else float('inf')
    accepted = bool(repeated_scatter and worst_scatter <= maximum_relative_scatter)
    result = {'frequency_hz': frequency,
              'offset_hz': frequency - float(plan['centre_hz']),
              'lo_centre_hz': float(plan['centre_hz']),
              'power_gain': power_gain,
              'voltage_gain': np.sqrt(power_gain),
              'worst_repeated_relative_scatter': worst_scatter,
              'maximum_relative_scatter': float(maximum_relative_scatter),
              'decimation_log2': int(plan['decimation_log2']),
              'fir_expected_state': plan['fir_expected_state'],
              'accepted': accepted}
    model = cic_power_response(frequency, plan)
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    offset = (frequency - plan['centre_hz']) / 1e3
    axes[0].plot(offset, 10*np.log10(power_gain), 'o-', label='measured')
    axes[0].plot(offset, 10*np.log10(model), '--', label='CIC prediction')
    axes[0].set(ylabel='Normalized power gain (dB)',
                title='Acquisition response: measured versus CIC model')
    axes[0].legend()
    axes[1].plot(offset, 10*np.log10(power_gain/model), 'o-', color=CLR_GREY)
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set(xlabel='Frequency relative to LO (kHz)',
                ylabel='Measured/model residual (dB)')
    plt.tight_layout(); plt.show()
    if not accepted:
        raise CalibrationError(
            'Swept-tone response is not repeatable: worst repeated-point scatter '
            'is %.3f%%, limit %.3f%%. Check generator stability, ZFL compression, '
            'ADC headroom and randomized repeat measurements before correction.' %
            (100*worst_scatter, 100*maximum_relative_scatter))
    return result



---
# 8 — Measure and compare the three noise baselines

**What this cell does.** It measures three distinct conditions: (1) native high-impedance IN1 with
the ZFL disconnected/off, (2) powered follower/ZFL in a safe reference condition, and (3) the
complete feedback-off resonator. It plots raw PSD, acquisition gain and response-corrected PSD.

**Why it is needed.** There is no attenuator or 50 ohm terminator between the ZFL-500LN+ output
and native high-impedance IN1. This maximises measured voltage sensitivity after the coil signal
was barely visible even with more than 100 analyser averages, but it also removes overload margin
and changes ZFL loading, gain flatness and reflection behaviour. The measured noise, response and
oscillation tests therefore define the usable chain rather than nominal 50 ohm gain. Comparing the three floors identifies
whether sensitivity is limited by the Red Pitaya, the powered amplifier chain, or the resonator
coupling. Different floors between the three conditions are expected and are not called drift.

**Hardware sequence.** `feedback_off()` clears OUT1 coefficients and both generators. Power the
ZFL down before changing its output connection. Because high-impedance IN1 is not the ZFL's nominal
50 ohm load, first inspect the powered-chain spectrum for oscillation or excess broadband power.
Restore the complete resonator only after the direct ZFL-to-IN1 cable is secure.
Do not alter the 30 dB OUT1 return pad during this test.

**Expected plots and interpretation.** If powered-chain noise is far above bare-IN1 noise,
the Red Pitaya is not the sensitivity limit. If the two are similar, additional analogue gain may
eventually be needed. If connecting the resonator produces a broad 100–800 kHz feature that remains
fixed under LO movement, it is a physical candidate. Narrow bins are treated as spurs.

**Pass criteria.** Each condition must be stationary within itself, unclipped and repeatable. The
cell reports chain/ADC and resonator/chain power ratios with uncertainties. A failure names the
condition, peak voltage, headroom, floor scatter and corrective action. Safe to rerun only after
feedback is zero and the ZFL has been sequenced safely.


In [ ]:
NOISE_CALIBRATION = {}
ACQUISITION_RESPONSE_CALIBRATION = None

def measure_noise_condition(label):
    feedback_off()
    records = acquire_passive_records()
    health = record_health(records)
    raw_spectrum = make_passive_spectrum(records)
    spectrum = correct_acquisition_response(
        raw_spectrum, INITIAL_ACQUISITION_PLAN,
        calibration=ACQUISITION_RESPONSE_CALIBRATION)
    search = (np.abs(spectrum['frequency_hz'] - CENTRE_HZ) <=
              DISCOVERY_SEARCH_HALF_WIDTH_HZ)
    valid_search = search & spectrum['response_correction_valid']
    median_floor = float(np.nanmedian(spectrum['psd_v2_per_hz'][valid_search]))
    individual_floor = np.nanmedian(
        spectrum['individual_psd_v2_per_hz'][:, search], axis=1)
    floor_relative_scatter = float(np.std(individual_floor, ddof=1) /
                                   max(np.mean(individual_floor), np.finfo(float).tiny))
    spur_ratio = float(np.nanmax(spectrum['psd_v2_per_hz'][valid_search]) /
                       max(median_floor, np.finfo(float).tiny))
    result = {'label': label, 'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
              'health': health, 'median_floor_v2_per_hz': median_floor,
              'floor_relative_scatter': floor_relative_scatter,
              'largest_bin_to_floor': spur_ratio, 'spectrum': spectrum,
              'records': records}
    NOISE_CALIBRATION[label] = result
    print(json.dumps({k: v for k, v in result.items() if k not in ('spectrum', 'records')},
                     indent=2, default=str))
    search = np.abs(spectrum['frequency_hz'] - CENTRE_HZ) <= DISCOVERY_SEARCH_HALF_WIDTH_HZ
    if not np.any(search & spectrum['response_correction_valid']):
        raise CalibrationError(
            'No trustworthy response-corrected bins remain inside 38.5–39.5 MHz. '
            'Run the swept-tone acquisition-response calibration or reduce the '
            'requested interval; do not fit the raw CIC-shaped spectrum.')
    frequency_khz = (spectrum['frequency_hz'] - CENTRE_HZ) / 1e3
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    axes[0].semilogy(frequency_khz, spectrum['raw_psd_v2_per_hz'], color=CLR_BLUE)
    axes[0].set(ylabel='Raw PSD (V²/Hz)', title=label + ': raw acquisition output')
    axes[1].plot(frequency_khz, 10*np.log10(spectrum['acquisition_power_gain']),
                 color=CLR_ORANGE)
    axes[1].set(ylabel='Acquisition gain (dB)', title=spectrum['response_source'])
    axes[2].semilogy(frequency_khz[search], spectrum['psd_v2_per_hz'][search],
                     color=CLR_GREEN)
    axes[2].axhline(median_floor, color=CLR_ORANGE, linestyle='--', label='median floor')
    axes[2].set(xlabel='Frequency relative to 39 MHz (kHz)',
                ylabel='Corrected PSD (V²/Hz)', title='Accepted 1 MHz search interval')
    axes[2].legend()
    for axis in axes:
        axis.axvline(-500, color='black', linestyle=':', linewidth=0.8)
        axis.axvline(+500, color='black', linestyle=':', linewidth=0.8)
    plt.tight_layout(); plt.show()
    if not health['accepted']:
        feedback_off()
        raise RuntimeError(
            'Noise condition %r was rejected after %d records: %s. '
            'Peak %.6g V; operating headroom %.6g V; RMS CV %.4f. '
            'Feedback and generators remain off.' %
            (label, len(records), '; '.join(health['rejection_reasons']),
             health['peak_v'], health['headroom_to_operating_guard_v'],
             health['rms_cv']))
    return result

# Run the condition that matches the present physical connection:
# adc_noise = measure_noise_condition('in1_high_impedance_zfl_disconnected')
# chain_noise = measure_noise_condition('powered_detection_chain')
# resonator_noise = measure_noise_condition('complete_resonator_feedback_off')
# noise_campaign = assess_noise_campaign(NOISE_CALIBRATION)
# print(json.dumps(noise_campaign, indent=2, default=str))


---
# 9 — Locate the physical resonance with three LO centres, then refine

**What this cell does.** It repeats the feedback-off acquisition with LO offsets of −200, 0 and
+200 kHz. Every spectrum is response-corrected and fitted only inside the same absolute
38.5–39.5 MHz interval. It compares the three fitted centres before allowing adaptive zoom.

**Why it is needed.** A real resonator stays at one absolute frequency when the digital LO moves.
A CIC/FIR edge, passband ripple or numerical edge artefact moves with the LO. The previous run
selected an edge near 42.902 MHz, reported Q≈18,000, and zoomed into it; this cell explicitly makes
that sequence impossible.

**Hardware.** Feedback and both generators remain zero. The connected passive resonator, direct
ZFL-to-IN1 high-impedance connection, amplifier settings and cables remain unchanged throughout
all three captures. A healthy OUT2 unity monitor may remain on.

**Expected plots.** Corrected spectra overlaid in absolute frequency, fitted centres for all LO
settings, the final Lorentzian/background fit, and residuals. Discovery uses 40 averages; after a
physical centre and broad Q are established, refinement uses 120 averages and fits only ±1.25
discovery linewidths around that centre. Its allowed Q remains between 0.5 and 2 times the
discovery estimate, preventing an unrelated narrow spike from replacing the broad resonator. The CIC dome should move relative to
absolute frequency while the physical peak remains fixed.

**Pass criteria.** All provisional fits lie within 38.5–39.5 MHz, have Q initially between 50 and
1000, positive finite background, adequate prominence and uncertainty, and agree in centre within
0.25 linewidth. A provisional feature closer than 1.5 linewidths to an edge is reported as a warning,
not discarded: agreement across three LO positions must first establish that it is physical. The
notebook then recentres and accepts only a fully covered Q-adapted fit.

**Failure meaning.** A moving centre indicates acquisition-filter structure. An edge warning means
the provisional parameters may be biased and must not be used until recentered. Extreme Q or one-bin prominence indicates a spur/numerical fit. The verbose
error lists every failed criterion. Safe to rerun with feedback off.


In [ ]:
BASELINE_HISTORY = []


def _search_mask(frequency_hz):
    frequency = np.asarray(frequency_hz, dtype=float)
    return np.abs(frequency - CENTRE_HZ) <= DISCOVERY_SEARCH_HALF_WIDTH_HZ



def fit_broad_noise_peak(frequency_hz, psd, q_range=DISCOVERY_Q_RANGE,
                         minimum_delta_bic=10.0, minimum_integrated_snr=5.0):
    """Robustly fit a broad Lorentzian in coloured noise while masking narrow spurs.

    The fit works in log-PSD, uses a soft-L1 loss, bounds linewidth using the
    configured Q range, and compares against a sloping background-only model.
    A narrow maximum can no longer initialise an arbitrarily high-Q result.
    """
    from scipy.optimize import least_squares
    from scipy.ndimage import median_filter
    f = np.asarray(frequency_hz, dtype=float)
    y = np.asarray(psd, dtype=float)
    good = np.isfinite(f) & np.isfinite(y) & (y > 0)
    f, y = f[good], y[good]
    if len(f) < 128:
        raise CalibrationError(
            'Broad-noise detection requires at least 128 positive finite bins; '
            'received %d after response and spur masks.' % len(f))
    order = np.argsort(f); f, y = f[order], y[order]
    spacing = float(np.median(np.diff(f))); span = float(f[-1]-f[0])
    centre_ref = 0.5*(f[0]+f[-1]); x = (f-centre_ref)/max(span, 1.0)
    min_linewidth = centre_ref/float(q_range[1])
    max_linewidth = centre_ref/float(q_range[0])
    spur_width = max(3, int(round(min_linewidth/max(spacing, 1e-30)/8.0)))
    if spur_width % 2 == 0:
        spur_width += 1
    local = median_filter(y, size=spur_width, mode='nearest')
    spur_mask = (y/local < 5.0) & (local/y < 5.0)
    f_fit, y_fit, x_fit = f[spur_mask], y[spur_mask], x[spur_mask]
    logy = np.log(y_fit)
    background_design = np.column_stack((np.ones(len(x_fit)), x_fit))
    # Use the legacy threshold explicitly: the board's older NumPy rejects a null rcond.
    background_coef = np.linalg.lstsq(background_design, logy, rcond=-1)[0]
    background_log = background_design.dot(background_coef)
    background = np.exp(background_log)
    excess = np.maximum(y_fit-background, 0.0)
    # Matched broad smoothing chooses a seed immune to isolated FFT bins.
    kernel_bins = max(5, int(round(min_linewidth/max(spacing, 1e-30))))
    kernel = np.ones(kernel_bins)/kernel_bins
    score = np.convolve(excess/background, kernel, mode='same')
    seed_f0 = float(f_fit[int(np.argmax(score))])
    seed_amp = max(float(np.percentile(excess, 95)), 0.05*float(np.median(background)))
    seed_linewidth = min(max(centre_ref/150.0, min_linewidth), max_linewidth)

    def model(parameters, frequency=f_fit, xx=x_fit):
        b0, b1, log_amp, f0, log_width = parameters
        bg = np.exp(b0+b1*xx)
        amp = np.exp(log_amp); width = np.exp(log_width)
        return bg + amp/(1.0+4.0*((frequency-f0)/width)**2)

    initial = [background_coef[0], background_coef[1], np.log(seed_amp),
               seed_f0, np.log(seed_linewidth)]
    lower = [background_coef[0]-5, -20, np.log(np.finfo(float).tiny),
             f[0], np.log(min_linewidth)]
    upper = [background_coef[0]+5, 20, np.log(max(y)*100),
             f[-1], np.log(min(max_linewidth, span))]
    tiny = np.finfo(float).tiny
    largest = np.finfo(float).max
    def log_model_residual(parameters):
        # Optimisers deliberately try poor intermediate parameters. Convert any
        # resulting non-finite trial model into a finite penalty without emitting
        # the misleading 'invalid value encountered in maximum' warning.
        trial = np.nan_to_num(model(parameters), nan=tiny,
                              posinf=largest, neginf=tiny)
        return np.log(np.clip(trial, tiny, largest))-logy
    result = least_squares(
        log_model_residual,
        initial, bounds=(lower, upper), loss='soft_l1', f_scale=0.15,
        max_nfev=30000)
    fitted_masked = np.nan_to_num(model(result.x), nan=tiny, posinf=largest, neginf=tiny)
    fitted_full = np.nan_to_num(model(result.x, f, x), nan=tiny, posinf=largest, neginf=tiny)
    residual_log = logy-np.log(np.clip(fitted_masked, tiny, largest))
    background_residual = logy-background_log
    rss = float(np.sum(residual_log**2)); rss0 = float(np.sum(background_residual**2))
    n = len(logy)
    bic = n*np.log(max(rss/n, np.finfo(float).tiny))+5*np.log(n)
    bic0 = n*np.log(max(rss0/n, np.finfo(float).tiny))+2*np.log(n)
    delta_bic = float(bic0-bic)
    amp = float(np.exp(result.x[2])); f0 = float(result.x[3])
    linewidth = float(np.exp(result.x[4])); floor_at_peak = float(
        np.exp(result.x[0]+result.x[1]*((f0-centre_ref)/max(span, 1.0))))
    bins_in_linewidth = max(linewidth/max(spacing, 1e-30), 1.0)
    integrated_snr = float((amp/floor_at_peak)*np.sqrt(bins_in_linewidth))
    accepted = bool(result.success and delta_bic >= minimum_delta_bic and
                    integrated_snr >= minimum_integrated_snr)
    reasons = []
    if not result.success:
        reasons.append('robust broad-peak optimizer did not converge')
    if delta_bic < minimum_delta_bic:
        reasons.append('resonance/background ΔBIC %.3f is below %.3f' %
                       (delta_bic, minimum_delta_bic))
    if integrated_snr < minimum_integrated_snr:
        reasons.append('integrated broad-feature SNR %.3f is below %.3f' %
                       (integrated_snr, minimum_integrated_snr))
    return {'accepted': accepted, 'resonator_state': 'passive_noise',
            'f0_hz': f0, 'linewidth_hz': linewidth, 'q': f0/linewidth,
            'noise_floor': floor_at_peak, 'slope': float(result.x[1]),
            'resonator_amplitude': amp, 'model': fitted_full,
            'residual': y-fitted_full, 'rss': rss,
            'delta_bic': delta_bic, 'integrated_snr': integrated_snr,
            'masked_spur_bins': int(np.count_nonzero(~spur_mask)),
            'f0_uncertainty_hz': linewidth/max(integrated_snr, 1.0),
            'linewidth_uncertainty_hz': linewidth/max(integrated_snr, 1.0),
            'broad_detection_reasons': reasons}

def fit_one_spectrum(spectrum, discovery=True, q_range=None,
                     fit_centre_hz=None, fit_half_width_hz=None):
    """Fit finite corrected samples in either the discovery or local interval."""
    frequency = np.asarray(spectrum['frequency_hz'], dtype=float)
    psd = np.asarray(spectrum['psd_v2_per_hz'], dtype=float)
    mask = np.isfinite(psd)
    if discovery:
        mask &= _search_mask(frequency)
    elif fit_centre_hz is not None and fit_half_width_hz is not None:
        local_half_width = float(fit_half_width_hz)
        if not np.isfinite(local_half_width) or local_half_width <= 0:
            raise CalibrationError('local fit half-width must be positive and finite')
        mask &= np.abs(frequency - float(fit_centre_hz)) <= local_half_width
    if np.count_nonzero(mask) < 64:
        raise CalibrationError(
            'Only %d finite bins remain in the selected resonance-fit interval; '
            'at least 64 are required. Check response calibration, LO centre and '
            'capture span before fitting.' %
            np.count_nonzero(mask))
    active_q_range = DISCOVERY_Q_RANGE if q_range is None else tuple(q_range)
    fit = fit_broad_noise_peak(frequency[mask], psd[mask], q_range=active_q_range)
    floor = float(fit['noise_floor'])
    prominence = (float(fit['resonator_amplitude']) / floor
                  if np.isfinite(floor) and floor > 0 else float('inf'))
    reasons = []
    low = CENTRE_HZ - DISCOVERY_SEARCH_HALF_WIDTH_HZ
    high = CENTRE_HZ + DISCOVERY_SEARCH_HALF_WIDTH_HZ
    if discovery and not low <= fit['f0_hz'] <= high:
        reasons.append('fitted centre %.6f MHz is outside %.6f..%.6f MHz' %
                       (fit['f0_hz']/1e6, low/1e6, high/1e6))
    edge_distance = min(fit['f0_hz'] - low, high - fit['f0_hz'])
    edge_margin_linewidths = edge_distance/max(fit['linewidth_hz'], np.finfo(float).tiny)
    edge_warning = None
    if discovery and edge_margin_linewidths < DISCOVERY_EDGE_MARGIN_LINEWIDTHS:
        # This first pass exists only to identify an LO-invariant candidate. A broad
        # real resonance can legitimately sit near the fixed discovery boundary. Do
        # not discard it here: require agreement across all three LO positions, then
        # recenter and assess the fully covered refined capture below.
        edge_warning = ('provisional discovery centre is %.2f linewidths from a search '
                        'edge; it must pass three-LO invariance and a recentered fit' %
                        edge_margin_linewidths)
    if not active_q_range[0] <= fit['q'] <= active_q_range[1]:
        reasons.append('broad-fit Q %.3g is outside active range %.3g..%.3g' %
                       (fit['q'], active_q_range[0], active_q_range[1]))
    elif (fit['q'] <= 1.02*active_q_range[0] or
          fit['q'] >= 0.98*active_q_range[1]):
        reasons.append('fitted Q %.3g is pinned near the active Q bound %.3g..%.3g; '
                       'the local single-resonance model is not trustworthy' %
                       (fit['q'], active_q_range[0], active_q_range[1]))
    if not np.isfinite(floor) or floor <= 0:
        reasons.append('fitted background floor is not finite and positive')
    if not np.isfinite(prominence) or prominence <= 0:
        reasons.append('fitted broad-feature amplitude/background is not positive and finite')
    reasons.extend(fit.get('broad_detection_reasons', []))
    if fit['f0_uncertainty_hz'] > 0.20 * fit['linewidth_hz']:
        reasons.append('centre uncertainty %.6g Hz exceeds 20%% of linewidth %.6g Hz' %
                       (fit['f0_uncertainty_hz'], fit['linewidth_hz']))
    fit['peak_prominence'] = prominence
    fit['edge_margin_linewidths'] = edge_margin_linewidths
    fit['edge_margin_warning'] = edge_warning
    fit['rejection_reasons'] = reasons
    fit['accepted'] = bool(fit.get('accepted', False) and not reasons)
    fit['fit_frequency_hz'] = frequency[mask]
    return fit


def assess_lo_invariance(results, maximum_centre_difference_linewidths=0.25):
    """Require a feature to remain fixed in absolute frequency as LO moves."""
    if len(results) < 3:
        raise CalibrationError(
            'LO-invariance requires at least three captures with distinct LO '
            'centres. One spectrum cannot distinguish a resonator from CIC/FIR structure.')
    centres = np.asarray([row['fit']['f0_hz'] for row in results], dtype=float)
    los = np.asarray([row['plan']['centre_hz'] for row in results], dtype=float)
    linewidth = float(np.median([row['fit']['linewidth_hz'] for row in results]))
    spread = float(np.max(centres) - np.min(centres))
    distinct_los = len(np.unique(los)) == len(los)
    accepted = bool(distinct_los and all(row['fit']['accepted'] for row in results) and
                    spread <= maximum_centre_difference_linewidths * linewidth)
    return {'accepted': accepted, 'centre_spread_hz': spread,
            'centre_spread_linewidths': spread/max(linewidth, np.finfo(float).tiny),
            'maximum_centre_difference_linewidths': float(maximum_centre_difference_linewidths),
            'lo_centres_hz': los, 'fitted_centres_hz': centres}


def _capture_discovery_at_lo(lo_hz, label):
    plan = dict(INITIAL_ACQUISITION_PLAN)
    plan['centre_hz'] = float(lo_hz)
    records = acquire_passive_records(plan=plan)
    health = record_health(records)
    raw = make_passive_spectrum(records, plan=plan)
    spectrum = correct_acquisition_response(
        raw, plan, calibration=ACQUISITION_RESPONSE_CALIBRATION)
    fit = fit_one_spectrum(spectrum, discovery=True)
    return {'label': label, 'timestamp_epoch_s': time.time(), 'health': health,
            'spectrum': spectrum, 'fit': fit, 'fits': {'none': fit}, 'plan': plan}


def measure_baseline(label='feedback_off', plan=None, averages=None):
    """Discover with three LO centres initially; accept explicit later plans singly."""
    feedback_off()
    if plan is not None:
        averages = REFINED_BASELINE_AVERAGES if averages is None else int(averages)
        records = acquire_passive_records(plan=plan, averages=averages)
        health = record_health(records)
        raw = make_passive_spectrum(records, plan=plan)
        spectrum = correct_acquisition_response(raw, plan, calibration=None)
        q_window = (max(20.0, plan['q_estimate']*LOCAL_Q_LOWER_FACTOR),
                    min(1.0e6, plan['q_estimate']*LOCAL_Q_UPPER_FACTOR))
        fit = fit_one_spectrum(
            spectrum, discovery=False, q_range=q_window,
            fit_centre_hz=plan['centre_hz'],
            fit_half_width_hz=LOCAL_FIT_LINEWIDTHS_EACH_SIDE*plan['linewidth_hz'])
        result = {'label': label, 'timestamp_epoch_s': time.time(),
                  'health': health, 'spectrum': spectrum, 'fit': fit,
                  'fits': {'none': fit}, 'plan': plan}
        if not health['accepted'] or not fit['accepted']:
            raise CalibrationError(
                'Planned feedback-off baseline %r failed: capture reasons=%s; '
                'fit reasons=%s. Feedback remains off.' %
                (label, '; '.join(health['rejection_reasons']) or 'none',
                 '; '.join(fit['rejection_reasons']) or 'underlying fitter rejected'))
        BASELINE_HISTORY.append(result)
        return result
    discoveries = []
    for offset in DISCOVERY_LO_OFFSETS_HZ:
        row = _capture_discovery_at_lo(CENTRE_HZ + offset,
                                       '%s_lo_%+.0fkHz' % (label, offset/1e3))
        discoveries.append(row)
        print('LO %.6f MHz -> fitted %.6f MHz, Q %.1f; accepted=%s%s' %
              (row['plan']['centre_hz']/1e6, row['fit']['f0_hz']/1e6,
               row['fit']['q'], row['fit']['accepted'],
               ('; WARNING: ' + row['fit']['edge_margin_warning'])
               if row['fit'].get('edge_margin_warning') else
               ('' if row['fit']['accepted'] else
                '; ' + '; '.join(row['fit']['rejection_reasons']))))
    invariance = assess_lo_invariance(discoveries)
    fig, axis = plt.subplots(figsize=(10, 4.5))
    for row in discoveries:
        search = _search_mask(row['spectrum']['frequency_hz'])
        axis.semilogy(row['spectrum']['frequency_hz'][search]/1e6,
                      row['spectrum']['psd_v2_per_hz'][search],
                      label='LO %.3f MHz' % (row['plan']['centre_hz']/1e6))
    axis.set(xlabel='Absolute frequency (MHz)', ylabel='Corrected PSD (V²/Hz)',
             title='LO-invariance test: a physical resonance stays fixed')
    axis.legend(); plt.tight_layout(); plt.show()
    if not invariance['accepted']:
        feedback_off()
        reasons = ['LO-invariance centre spread %.6g Hz = %.3f linewidths; limit %.3f' %
                   (invariance['centre_spread_hz'], invariance['centre_spread_linewidths'],
                    invariance['maximum_centre_difference_linewidths'])]
        for row in discoveries:
            reasons.extend('%s: %s' % (row['label'], reason)
                           for reason in row['fit']['rejection_reasons'])
        raise CalibrationError(
            'Physical-resonance discovery was rejected. %s. The notebook will '
            'not recenter or enable feedback. Check swept-tone response, spurs, '
            'search centre and analogue resonance before retrying.' % '; '.join(reasons))
    discovery_fit = discoveries[1]['fit']
    refined_plan = acquisition_plan(discovery_fit['f0_hz'], discovery_fit['q'])
    refined_averages = REFINED_BASELINE_AVERAGES if averages is None else int(averages)
    refined_records = acquire_passive_records(plan=refined_plan, averages=refined_averages)
    refined_health = record_health(refined_records)
    refined_raw = make_passive_spectrum(refined_records, plan=refined_plan)
    refined_spectrum = correct_acquisition_response(
        refined_raw, refined_plan, calibration=None)
    refined_q_window = (max(20.0, refined_plan['q_estimate']*LOCAL_Q_LOWER_FACTOR),
                        min(1.0e6, refined_plan['q_estimate']*LOCAL_Q_UPPER_FACTOR))
    refined_fit = fit_one_spectrum(
        refined_spectrum, discovery=False, q_range=refined_q_window,
        fit_centre_hz=discovery_fit['f0_hz'],
        fit_half_width_hz=LOCAL_FIT_LINEWIDTHS_EACH_SIDE*discovery_fit['linewidth_hz'])
    result = {'label': label, 'timestamp_epoch_s': time.time(),
              'health': refined_health, 'spectrum': refined_spectrum,
              'fit': refined_fit, 'fits': {'none': refined_fit},
              'plan': refined_plan, 'discoveries': discoveries,
              'lo_invariance': invariance}
    BASELINE_HISTORY.append(result)
    print('ACCEPTED physical resonance: f0 %.6f MHz, Q %.1f, linewidth %.3f kHz, '
          'prominence %.2f' %
          (refined_fit['f0_hz']/1e6, refined_fit['q'],
           refined_fit['linewidth_hz']/1e3, refined_fit['peak_prominence']))
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    fit_frequency = refined_fit['fit_frequency_hz']
    axes[0].semilogy(fit_frequency/1e6, refined_spectrum['psd_v2_per_hz'][
        np.isfinite(refined_spectrum['psd_v2_per_hz'])], '.', markersize=2,
        color=CLR_BLUE, label='response-corrected PSD')
    axes[0].semilogy(fit_frequency/1e6, refined_fit['model'], color=CLR_ORANGE,
                     label='Lorentzian + background')
    axes[0].set(ylabel='PSD (V²/Hz)', title='Refined accepted resonator fit')
    axes[0].legend()
    axes[1].plot(fit_frequency/1e6,
                 refined_spectrum['psd_v2_per_hz'][np.isfinite(refined_spectrum['psd_v2_per_hz'])]
                 - refined_fit['model'], color=CLR_GREY)
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set(xlabel='Absolute frequency (MHz)', ylabel='Fit residual (V²/Hz)')
    plt.tight_layout(); plt.show()
    if not refined_health['accepted'] or not refined_fit['accepted']:
        feedback_off()
        raise CalibrationError(
            'Refined baseline failed after LO-invariant discovery: health reasons=%s; '
            'fit reasons=%s. Feedback remains off.' %
            ('; '.join(refined_health['rejection_reasons']) or 'none',
             '; '.join(refined_fit['rejection_reasons']) or 'underlying fitter rejected'))
    return result

# baseline = measure_baseline()


---
# 10 — Spectrum-analyser cross-check

**What this cell does.** It loads a spectrum-analyser CSV, converts analyser-bin power using the
recorded RBW and cable settings, compares centre and Q with the accepted Red Pitaya result, and
plots the residual difference. It does not control the analyser or alter feedback.

**Why it is needed.** The analyser is the independent check on whether the broad Red Pitaya dome is
physical. A CIC feature moves with the Red Pitaya LO and is absent from the direct analyser branch.
If the analyser is connected to OUT2, remember that OUT2 reproduces digitised IN1 before the
measurement mixer/CIC/FIR, so it still does not display the acquisition-filter response.

**Hardware and settings.** This cell performs no instrument control. Export with 50 ohm input, fixed
attenuation, preamp off initially, RMS detector, recorded RBW/VBW/averaging and the same cable. Do
not change these after the configuration fingerprint is stored.

**Expected result and pass criteria.** Physical resonance centre and Q agree within combined
uncertainty. A smooth Red-Pitaya-only residual that follows LO offset identifies acquisition response,
not resonator physics. Failure reports whether centre, Q, scaling or analyser identity disagrees.
Safe to rerun because it only reads a file and performs calculations.


In [ ]:
def load_analyser_trace(csv_path, frequency_column=0, power_column=1,
                          power_unit='dbm'):
    values = np.loadtxt(str(csv_path), delimiter=',', comments='#')
    frequency = values[:, frequency_column]
    shown = values[:, power_column]
    if power_unit.lower() == 'dbm':
        power = 1e-3 * 10.0 ** (shown / 10.0)
    elif power_unit.lower() == 'w':
        power = shown
    else:
        raise ValueError('power_unit must be dbm or W')
    return frequency, power

def compare_analyser(red_pitaya_spectrum, analyser_csv):
    if analyser_configuration_fingerprint(ANALYSER) != ANALYSER_FINGERPRINT:
        raise RuntimeError('analyser configuration changed; recalibration required')
    af, ap = load_analyser_trace(analyser_csv)
    rp_fit = fit_one_spectrum(red_pitaya_spectrum)
    sa_fit = fit_passive_resonator(af, ap)
    comparison = {
        'f0_difference_hz': rp_fit['f0_hz'] - sa_fit['f0_hz'],
        'q_fractional_difference': rp_fit['q'] / sa_fit['q'] - 1.0,
        'median_branch_power_ratio': float(np.median(red_pitaya_spectrum['psd_v2_per_hz']) /
                                           max(np.median(ap), np.finfo(float).tiny)),
        'analyser_fingerprint': ANALYSER_FINGERPRINT,
    }
    print(json.dumps(comparison, indent=2))
    return comparison


---
# Optional — Measure OUT1 DC offset on a dummy load

**Purpose.** Replace the initial digital-zero/open-gap assumption with an external measurement of
OUT1 through the actual 30 dB pad.

**Hardware.** Disconnect the air-gap pin; use a DMM or scope and a safe 50 ohm dummy load. This is
optional because the open air gap blocks DC current into the coil, but it is strongly recommended
before reducing attenuation or changing the gap. **Expected result.** Zero-command,
coefficient-dependent and startup/shutdown excursions below configured limits. **Pass.** Each value
plus three uncertainties is inside its limit. **Failure.** `full_shutdown()` and do not reconnect
OUT1. **Safe to rerun.** Yes, on the dummy load only.


In [ ]:
def record_out1_dc_commissioning(zero_offset_v, maximum_coefficient_offset_v,
                                 startup_shutdown_peak_v, instrument,
                                 uncertainty_v, shutdown_verified):
    global OUT1_DC_COMMISSIONING
    values = [float(zero_offset_v), float(maximum_coefficient_offset_v),
              float(startup_shutdown_peak_v), float(uncertainty_v)]
    if not all(np.isfinite(values)) or values[3] <= 0:
        raise ValueError('all DC commissioning values must be finite; uncertainty must be positive')
    accepted = bool(abs(values[0]) + 3 * values[3] <= OUT1_DC_OFFSET_LIMIT_V and
                    abs(values[1]) + 3 * values[3] <= OUT1_DC_OFFSET_LIMIT_V and
                    abs(values[2]) + 3 * values[3] <= OUT1_TRANSIENT_LIMIT_VPK and
                    shutdown_verified and str(instrument).strip())
    record = {'zero_offset_v': values[0], 'maximum_coefficient_offset_v': values[1],
              'startup_shutdown_peak_v': values[2], 'instrument': str(instrument),
              'uncertainty_v': values[3], 'shutdown_verified': bool(shutdown_verified),
              'accepted': accepted, 'return_attenuation_db_nominal': 30.0,
              'airgap_was_disconnected': True, 'timestamp_epoch_s': time.time()}
    OUT1_DC_COMMISSIONING = _seal(record)
    print(json.dumps(record, indent=2))
    if not accepted:
        full_shutdown()
        raise RuntimeError('OUT1 DC/transient commissioning failed; do not connect the air-gap pin')
    return OUT1_DC_COMMISSIONING

# OUT1_DC_COMMISSIONING = record_out1_dc_commissioning(...)


---
# 11 — Commission the unity IN1-to-OUT2 campaign monitor

**Purpose.** enables the existing FPGA input monitor at shift 0 and verifies generator-B zero, register readback, analyser configuration, live IN1 headroom, and the sticky monitor-clipping flag. No FPGA changes are made.

**Wiring.** OUT2 is an output—connect it only to a 50 ohm spectrum-analyser input, never another source. Start with analyser preamp off, conservative internal attenuation, and a reference level above the declared possible OUT2 power. **Why.** OUT2 is useful for live observation but is not independent; it includes ADC quantisation, FPGA scaling, DAC noise and reconstruction response. **Pass.** a stable physical trace, no overload/clipping, and electrical readback and level checks. A failure invokes `full_shutdown()`. Safe to rerun after correcting the analyser setup.


In [ ]:
def out2_monitor_status():
    state = ps.readback()
    return {'source_code': state['monitor'] & 7,
            'shift': (state['monitor'] >> 3) & 7,
            'register': state['monitor'],
            'clipped': ps.monitor_clipped(),
            'analyser_fingerprint': analyser_configuration_fingerprint(ANALYSER)}

def out2_monitor_enable(shift=0):
    global OUT2_MONITOR_CALIBRATION
    try:
        if int(shift) != 0:
            raise RuntimeError('only unity shift=0 is permitted in this campaign')
        configuration_report()
        if str(ANALYSER.get('preamp_state', '')).lower() != 'off':
            raise RuntimeError('commission OUT2 with analyser preamplifier off')
        if float(ANALYSER['maximum_safe_input_dbm']) < OUT2_DECLARED_MAX_DBM:
            raise RuntimeError('analyser safe-input rating is below the declared OUT2 maximum')
        if float(ANALYSER['reference_level_dbm']) < OUT2_DECLARED_MAX_DBM:
            raise RuntimeError('analyser reference level is too low for initial commissioning')
        feedback_off()
        gen.assert_b_zero()
        live_peak_v = sc.peak_frac(4096) * RP_LIMITS['input_lv_full_scale_vpk']
        if live_peak_v >= ADC_OPERATING_GUARD_VPK:
            raise RuntimeError('IN1 is too close to full scale for OUT2 monitoring')
        ps.monitor('input', shift=0)
        status = out2_monitor_status()
        if status['source_code'] != 1 or status['shift'] != 0 or status['clipped']:
            raise RuntimeError('OUT2 monitor readback or clipping check failed')
        OUT2_MONITOR_CALIBRATION = _seal({
            'accepted': True, 'unity_shift': 0, 'live_input_peak_v': live_peak_v,
            'analyser': copy.deepcopy(ANALYSER),
            'analyser_fingerprint': analyser_configuration_fingerprint(ANALYSER),
            'timestamp_epoch_s': time.time(),
            'independent_resonator_measurement': False})
        print('PASS: OUT2 is a unity copy of digitised IN1; campaign monitor remains active.')
        return OUT2_MONITOR_CALIBRATION
    except BaseException:
        full_shutdown()
        raise

def compare_out2_analyser_trace(frequency_hz, power_dbm, rp_spectrum,
                                tolerance_db=3.0):
    frequency_hz = np.asarray(frequency_hz, dtype=float)
    power_dbm = np.asarray(power_dbm, dtype=float)
    if frequency_hz.shape != power_dbm.shape or frequency_hz.size < 8:
        raise ValueError('analyser trace must contain matching frequency/power arrays')
    if analyser_configuration_fingerprint(ANALYSER) != ANALYSER_FINGERPRINT:
        raise RuntimeError('analyser configuration changed since fingerprinting')
    rbw = float(ANALYSER['rbw_hz'])
    cable_loss = float(ANALYSER['cable_loss_db'])
    measured_v2_hz = 10 ** ((power_dbm + cable_loss - 30.0) / 10.0) * 50.0 / rbw
    expected = np.interp(frequency_hz, rp_spectrum['frequency_hz'],
                         rp_spectrum['psd_v2_per_hz'])
    residual_db = 10.0 * np.log10(np.maximum(measured_v2_hz, np.finfo(float).tiny) /
                                  np.maximum(expected, np.finfo(float).tiny))
    median_error = float(np.median(residual_db))
    accepted = bool(abs(median_error) <= float(tolerance_db))
    plt.figure(figsize=(9, 4))
    plt.plot((frequency_hz - CENTRE_HZ) / 1e3, residual_db, color=CLR_BLUE)
    plt.axhline(0.0, color='black'); plt.axhline(tolerance_db, color=CLR_RED, linestyle='--')
    plt.axhline(-tolerance_db, color=CLR_RED, linestyle='--')
    plt.xlabel('Frequency relative to 39 MHz (kHz)'); plt.ylabel('OUT2/RP difference (dB)')
    plt.title('Spectrum-analyser agreement with digitised IN1'); plt.show()
    return {'accepted': accepted, 'median_error_db': median_error,
            'tolerance_db': float(tolerance_db),
            'independent_resonator_measurement': False}

# OUT2_MONITOR_CALIBRATION = out2_monitor_enable()


---
# 12 — Calibrate observable feedback from the smallest coefficient

**Purpose.** defines and uses its own guarded feedback-point measurement, then applies four cardinal phases at one coefficient count, followed by 2, 4, 8 and upward. Every active phase and feedback-off bracket uses 120 averaged spectra and the same narrow, discovery-anchored fit interval. Its Q and frequency change are measured relative to the midpoint of the surrounding brackets. This reduces random fit scatter and prevents ordinary baseline drift or neighbouring resonances from being attributed to feedback. It stops at the first statistically observable resonator change and measures the effective amplification of the unknown external circuit without assuming its gain. The cell has no dependency on a helper defined in a later cell.

**Hardware.** yes, OUT1 feedback. Required state: live IN1 safety passed, the digital-zero/open-gap offset record, fixed 30 dB pad, open stable air gap, and commissioned OUT2 monitor. **Expected result.** one diagnostic line per count and phase, followed by plots of Q and centre changes versus coefficient count. **Pass.** the selected rung is observable, remains below 10% Q change and half a linewidth shift, and both adjacent feedback-off spectra agree within 10% Q and 0.25 linewidth. An excessive active effect or failed recovery invokes full shutdown and reports count, gain, phase, fitted Q/frequency, bracketed reference and both limits. Do not increase gain after this stop. **Safe to rerun.** only after fresh feedback-off baselines and recommissioning OUT2 if full shutdown disabled it.

**Stopping safely.** Run `feedback_off()` to stop OUT1 feedback while retaining a healthy OUT2 monitor. Run `out2_monitor_off()` to disable and verify only the monitor. Run `disable_everything()` (or `full_shutdown()`) at any time to clear both generators, OUT1 coefficients, OUT2 monitoring and acquisition, with register readback verification. If Python is interrupted before confirmation, run `disable_everything()` immediately.


In [ ]:
SAFE_PROBE_CALIBRATION = None

def _measure_feedback_point(gain, phase_deg, baseline_spectrum, plan,
                            averages=SAFE_PROBE_AVERAGES):
    """Apply one guarded feedback point and return its passive-spectrum fit."""
    gen.off()
    input_peak_fraction = sc.peak_frac(4096)
    input_peak_v = input_peak_fraction * RP_LIMITS['input_lv_full_scale_vpk']
    predicted_out1_vpk = (input_peak_fraction * abs(float(gain)) *
                            RP_LIMITS['output_full_scale_vpk_into_50ohm'])
    if input_peak_v >= ADC_OPERATING_GUARD_VPK:
        raise RuntimeError('IN1 live peak %.6g V reached the %.6g V operating guard before feedback; all outputs remain off' %
                           (input_peak_v, ADC_OPERATING_GUARD_VPK))
    if predicted_out1_vpk >= OUT1_OPERATING_GUARD_VPK:
        raise RuntimeError('conservative OUT1 prediction %.6g V reached the %.6g V guard; all outputs remain off' %
                           (predicted_out1_vpk, OUT1_OPERATING_GUARD_VPK))
    ps.config_delay(CENTRE_HZ)
    coefficients = ps.set(float(gain), float(phase_deg))
    if (ps.readback()['ci'], ps.readback()['cq']) != tuple(coefficients):
        raise RuntimeError('feedback coefficient readback failed')
    records = acquire_passive_records(plan=plan, averages=int(averages))
    monitor = out2_monitor_status()
    if monitor['source_code'] != 1 or monitor['shift'] != 0 or monitor['clipped']:
        full_shutdown()
        raise RuntimeError('OUT2 monitor disabled, shifted, or clipped during feedback; full shutdown applied')
    gen.assert_b_zero()
    live_peak_v = sc.peak_frac(4096) * RP_LIMITS['input_lv_full_scale_vpk']
    if live_peak_v >= ADC_OPERATING_GUARD_VPK:
        raise RuntimeError('IN1 live peak %.6g V reached the %.6g V guard with feedback active' %
                           (live_peak_v, ADC_OPERATING_GUARD_VPK))
    spectrum = make_passive_spectrum(records, plan=plan)
    instability = detect_self_oscillation(spectrum['frequency_hz'],
                                          spectrum['psd_v2_per_hz'],
                                          baseline_spectrum['psd_v2_per_hz'])
    health = record_health(records)
    if instability['unsafe'] or not health['accepted']:
        raise RuntimeError('feedback point rejected for instability or capture health: instability=%r; capture reasons=%s' %
                           (instability, '; '.join(health['rejection_reasons']) or 'none'))
    q_window = (max(20.0, plan['q_estimate']*LOCAL_Q_LOWER_FACTOR),
                min(1.0e6, plan['q_estimate']*LOCAL_Q_UPPER_FACTOR))
    fit = fit_one_spectrum(
        spectrum, discovery=False, q_range=q_window,
        fit_centre_hz=plan['centre_hz'],
        fit_half_width_hz=LOCAL_FIT_LINEWIDTHS_EACH_SIDE*plan['linewidth_hz'])
    if not fit['accepted']:
        raise RuntimeError('feedback spectrum fit rejected: %s' %
                           ('; '.join(fit['rejection_reasons']) or 'underlying broad-feature detector rejected it'))
    return {'gain': float(gain), 'phase_deg': float(phase_deg),
            'coefficients': coefficients, 'fit': fit,
            'spectrum': spectrum, 'instability': instability,
            'input_peak_v_before': input_peak_v,
            'predicted_out1_vpk_bound': predicted_out1_vpk}

def calibrate_safe_probe_gain(coefficient_counts=None,
                              observable_q_fraction=0.02,
                              observable_shift_linewidths=0.05,
                              maximum_q_fraction=0.10,
                              maximum_shift_linewidths=0.50):
    """Find the first observable effective-loop gain, strictly low to high."""
    if not LIVE_INPUT_SAFETY.get('accepted', False):
        raise RuntimeError('live IN1 safety measurement has not passed')
    if OUT1_DC_COMMISSIONING is None or not OUT1_DC_COMMISSIONING.get('accepted', False):
        raise RuntimeError('pass the OUT1 dummy-load DC/transient commissioning first')
    if OUT2_MONITOR_CALIBRATION is None or not OUT2_MONITOR_CALIBRATION.get('accepted', False):
        raise RuntimeError('commission the unity OUT2 campaign monitor first')
    counts = (OUT1_INITIAL_COEFFICIENT_COUNTS if coefficient_counts is None
              else tuple(sorted({int(value): None for value in coefficient_counts})))
    if not counts or counts[0] < 1 or counts[-1] > int(OUT1_MAX_INITIAL_GAIN * 16384):
        raise ValueError('probe coefficient counts exceed the initial safe range')
    phases = (0.0, 90.0, 180.0, 270.0)
    rows = []
    selected = None
    try:
        for count in counts:
            gain = count / 16384.0
            before = measure_baseline('probe_before_%d' % count)
            effects = []
            for phase in phases:
                base_before = before['fits']['none']
                point = _measure_feedback_point(
                    gain, phase, before['spectrum'], before['plan'],
                    averages=SAFE_PROBE_AVERAGES)
                feedback_off()
                # Bracket every active point. This separates feedback from a drifting
                # or noisy single baseline and proves recovery before the next phase.
                after = measure_baseline('probe_after_%d_phase_%03d' %
                                         (count, int(phase)), plan=before['plan'],
                                         averages=SAFE_PROBE_AVERAGES)
                base_after = after['fits']['none']
                reference_q = 0.5 * (base_before['q'] + base_after['q'])
                reference_f0 = 0.5 * (base_before['f0_hz'] + base_after['f0_hz'])
                reference_linewidth = 0.5 * (base_before['linewidth_hz'] +
                                              base_after['linewidth_hz'])
                baseline_shift = (abs(base_after['f0_hz'] - base_before['f0_hz']) /
                                  max(reference_linewidth, np.finfo(float).tiny))
                baseline_q_change = abs(base_after['q']/base_before['q'] - 1.0)
                recovered = bool(baseline_shift <= 0.25 and baseline_q_change <= 0.10)
                q_fraction = abs(point['fit']['q'] / reference_q - 1.0)
                shift_linewidths = (abs(point['fit']['f0_hz'] - reference_f0) /
                                    max(reference_linewidth, np.finfo(float).tiny))
                effect = {'phase_deg': phase, 'q_fraction': q_fraction,
                          'shift_linewidths': shift_linewidths,
                          'baseline_shift_linewidths': baseline_shift,
                          'baseline_q_fraction': baseline_q_change,
                          'recovered': recovered, 'point': point}
                effects.append(effect)
                print('  count %d phase %3.0f deg: |dQ/Q| %.4f; shift %.4f linewidths; '
                      'off recovery dQ/Q %.4f, df %.4f linewidths' %
                      (count, phase, q_fraction, shift_linewidths,
                       baseline_q_change, baseline_shift))
                if not recovered:
                    full_shutdown()
                    raise RuntimeError(
                        'SAFE PROBE ABORTED at count %d (gain %.8g), phase %.1f deg: '
                        'feedback-off bracket did not recover. Before/after f0 were '
                        '%.9g/%.9g Hz and Q %.6g/%.6g; drift was %.4f linewidths and '
                        '%.2f%% in Q (limits 0.25 linewidth and 10%%). Full shutdown '
                        'was applied. Repeat feedback-off baselines before retrying.' %
                        (count, gain, phase, base_before['f0_hz'], base_after['f0_hz'],
                         base_before['q'], base_after['q'], baseline_shift,
                         100.0*baseline_q_change))
                if q_fraction > maximum_q_fraction or shift_linewidths > maximum_shift_linewidths:
                    full_shutdown()
                    raise RuntimeError(
                        'SAFE PROBE ABORTED at count %d (gain %.8g), phase %.1f deg. '
                        'Measured feedback point: f0 %.9g Hz, Q %.6g. Bracketed '
                        'feedback-off reference: f0 %.9g Hz, Q %.6g, linewidth %.6g Hz. '
                        'The apparent change was |dQ/Q| %.4f (limit %.4f) and shift '
                        '%.4f linewidths (limit %.4f). Full shutdown was applied. '
                        'Do not increase gain; inspect the printed phase result and '
                        'repeat passive baselines before deciding whether this was real feedback.' %
                        (count, gain, phase, point['fit']['f0_hz'], point['fit']['q'],
                         reference_f0, reference_q, reference_linewidth, q_fraction,
                         maximum_q_fraction, shift_linewidths, maximum_shift_linewidths))
                before = after
            max_q = max(item['q_fraction'] for item in effects)
            max_shift = max(item['shift_linewidths'] for item in effects)
            row = {'count': count, 'gain': gain, 'max_q_fraction': max_q,
                   'max_shift_linewidths': max_shift,
                   'recovered': all(item['recovered'] for item in effects),
                   'phase_effects': effects,
                   'accepted': True}
            rows.append(row)
            print('count %4d gain %.7f: max |dQ/Q| %.4f, max shift %.4f linewidths' %
                  (count, gain, max_q, max_shift))
            if max_q >= observable_q_fraction or max_shift >= observable_shift_linewidths:
                selected = row
                break
    except BaseException:
        feedback_off()
        raise
    finally:
        feedback_off()
    if selected is None:
        raise RuntimeError('no observable feedback within the initial safe coefficient range')
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    x = [row['count'] for row in rows]
    axes[0].plot(x, [row['max_q_fraction'] for row in rows], 'o-', color=CLR_BLUE)
    axes[0].axhline(maximum_q_fraction, color=CLR_RED, linestyle='--')
    axes[0].set(xscale='log', xlabel='Coefficient counts', ylabel='Maximum |dQ/Q|',
                title='Effective damping response')
    axes[1].plot(x, [row['max_shift_linewidths'] for row in rows], 'o-', color=CLR_ORANGE)
    axes[1].axhline(maximum_shift_linewidths, color=CLR_RED, linestyle='--')
    axes[1].set(xscale='log', xlabel='Coefficient counts', ylabel='Maximum shift (linewidths)',
                title='Effective reactive response')
    plt.tight_layout(); plt.show()
    result = {'accepted': True, 'selected_gain': selected['gain'],
              'selected_count': selected['count'], 'rows': rows,
              'coil_voltage_directly_measured': False,
              'airgap_transfer_directly_measured': False}
    print('Selected first observable gain:', selected['gain'])
    return result

# Emergency stop at any time: disable_everything()
# Disable only the spectrum-analyser copy: out2_monitor_off()
# Run the calibration after the preceding live checks:
# SAFE_PROBE_CALIBRATION = calibrate_safe_probe_gain()


---
# 13 — Low-gain phase sweep

**Purpose.** Measure a complete phase circle at the first observable coefficient while separating
feedback response from drift.

**Hardware.** OUT1 feedback is active one point at a time; generator drive remains off and OUT2
remains the unity monitor. **Expected result.** Smooth sinusoidal Q and centre responses. **Pass.**
Every point has accepted before/after baselines, stable headroom and no carrier or broadband
instability. **Failure.** Feedback is disabled; monitor faults invoke full shutdown. **Safe to
rerun.** Yes after baseline recovery.


In [ ]:
PHASE_RESULTS = []

def run_low_gain_phase_sweep(gain=None, phases_deg=None, seed=1):
    if SAFE_PROBE_CALIBRATION is None or not SAFE_PROBE_CALIBRATION.get('accepted', False):
        raise RuntimeError('run calibrate_safe_probe_gain() before the phase sweep')
    gain = SAFE_PROBE_CALIBRATION['selected_gain'] if gain is None else float(gain)
    if not 0 < gain <= OUT1_MAX_INITIAL_GAIN:
        raise ValueError('initial phase-sweep gain exceeds the commissioned range')
    phases = np.arange(0.0, 360.0, 15.0) if phases_deg is None else np.asarray(phases_deg)
    order = np.random.RandomState(seed).permutation(len(phases))
    output = []
    try:
        for index in order:
            before = measure_baseline('before_phase_%g' % phases[index])
            base_fit = before['fit']
            point = _measure_feedback_point(gain, phases[index],
                                            before['spectrum'], before['plan'])
            feedback_off()
            after = measure_baseline('after_phase_%g' % phases[index])
            after_fit = after['fit']
            drift_hz = after_fit['f0_hz'] - base_fit['f0_hz']
            if abs(drift_hz) > 0.25 * base_fit['linewidth_hz']:
                raise RuntimeError('feedback-off bracket drift exceeded 0.25 linewidth')
            point.update({'before': base_fit, 'after': after_fit, 'drift_hz': drift_hz})
            output.append(point)
    except BaseException:
        feedback_off()
        raise
    finally:
        feedback_off()
    PHASE_RESULTS[:] = output
    return output

# phase_results = run_low_gain_phase_sweep()


---
# 14 — Identify the four physical phase axes

**Purpose.** Fit Q and frequency versus phase as sinusoids with an arbitrary loop offset.

**Hardware.** None. **Expected result.** Regenerative, degenerative and both reactive axes with
uncertainties. **Pass.** At least 12 accepted points and damping/reactive axes sufficiently
orthogonal. **Failure.** Repeat the phase campaign; do not infer axes by eye. **Safe to rerun.** Yes.


In [ ]:
def calibrate_phase_axes(results):
    if len(results) < 12:
        raise CalibrationError('at least 12 accepted phase points are required')
    phase = np.deg2rad([row['phase_deg'] for row in results])
    design = np.column_stack((np.ones(len(phase)), np.cos(phase), np.sin(phase)))
    q = np.array([row['fit']['q'] for row in results])
    f0 = np.array([row['fit']['f0_hz'] for row in results])
    # rcond=-1 is compatible with both the Red Pitaya's older NumPy and current NumPy.
    qcoef = np.linalg.lstsq(design, q, rcond=-1)[0]
    fcoef = np.linalg.lstsq(design, f0, rcond=-1)[0]
    regenerative = np.degrees(np.arctan2(qcoef[2], qcoef[1])) % 360.0
    positive_shift = np.degrees(np.arctan2(fcoef[2], fcoef[1])) % 360.0
    separation = ((positive_shift - regenerative + 180.0) % 360.0) - 180.0
    q_resid = q - design.dot(qcoef); f_resid = f0 - design.dot(fcoef)
    phase_uncertainty = max(1.0, np.degrees(np.std(q_resid) /
        max(np.hypot(qcoef[1], qcoef[2]), np.finfo(float).tiny)))
    axes = {'regenerative_deg': regenerative,
            'degenerative_deg': (regenerative + 180.0) % 360.0,
            'positive_shift_deg': positive_shift,
            'negative_shift_deg': (positive_shift + 180.0) % 360.0,
            'uncertainty_deg': phase_uncertainty,
            'orthogonality_error_deg': abs(abs(separation) - 90.0)}
    if axes['orthogonality_error_deg'] > max(10.0, 3.0 * phase_uncertainty):
        raise CalibrationError('Q and shift axes are not sufficiently orthogonal')
    print(json.dumps(axes, indent=2))
    return axes

# PHASE_AXES = calibrate_phase_axes(PHASE_RESULTS)


---
# 15 — Monotonic gain ladders and paired K

**Purpose.** Measure proportionality, paired K, phase stability and recovery across gain. Absolute
gain increases monotonically; phase order is randomised only within each rung.

**Hardware.** OUT1 feedback is active one point at a time. **Expected result.** Linear Q/shift
response with compatible opposite-phase and joint estimates. **Pass.** Every rung has feedback-off
brackets, K <= 0.3, no clipping/oscillation and baseline recovery. **Failure.** Stop at the last
accepted rung; extrapolation is forbidden. **Safe to rerun.** Yes after recovery.


In [ ]:
GAIN_RESULTS = []

def run_gain_ladder(gains, phases, seed=2):
    gains = np.asarray(gains, dtype=float)
    if np.any(gains < 0) or np.max(gains) > 0.30:
        raise ValueError('commanded gain ladder must remain within 0..0.30 initially')
    rng = np.random.RandomState(seed)
    jobs = []
    for gain in sorted(gains):
        rung = [(float(gain), str(name), float(phi)) for name, phi in phases.items()]
        jobs.extend([rung[i] for i in rng.permutation(len(rung))])
    output = []
    try:
        for gain, name, phase in jobs:
            before = measure_baseline('before_%s_%g' % (name, gain))
            campaign_plan = acquisition_plan(
                before['fit']['f0_hz'],
                before['fit']['q'],
                FEEDBACK_MAX_EXPECTED_SHIFT_HZ)
            before = measure_baseline('wide_before_%s_%g' % (name, gain),
                                      plan=campaign_plan)
            point = _measure_feedback_point(gain, phase,
                                            before['spectrum'], before['plan'])
            feedback_off()
            after = measure_baseline('after_%s_%g' % (name, gain))
            point.update({'axis': name,
                          'baseline_q': 0.5 * (before['fit']['q'] +
                                              after['fit']['q']),
                          'baseline_f0_hz': 0.5 * (before['fit']['f0_hz'] +
                                                  after['fit']['f0_hz'])})
            output.append(point)
    except BaseException:
        feedback_off()
        raise
    finally:
        feedback_off()
    GAIN_RESULTS[:] = output
    return output

def report_paired_k(positive_q, opposite_q):
    value = paired_k(positive_q, opposite_q)
    if abs(value) > ELECTRODE_MAX_K:
        raise CalibrationError('measured K exceeds the initial 0.30 limit')
    print('Paired K = %.5f' % value)
    return value

# axes = {'regenerative': PHASE_AXES['regenerative_deg'],
#         'degenerative': PHASE_AXES['degenerative_deg'],
#         'positive_shift': PHASE_AXES['positive_shift_deg'],
#         'negative_shift': PHASE_AXES['negative_shift_deg']}
# gain_results = run_gain_ladder([0.0, 0.01, 0.02, 0.03], axes)


---
# 16 — Effective-loop linearity, hysteresis and recovery

**Purpose.** evaluates whether the observed resonator changes are proportional, phase-stable, non-hysteretic, and reversible after feedback is removed. This is the in-situ evidence that feedback reached the air-gap coil path; it is not a direct coil-voltage measurement.

**Hardware.** none. Required input: accepted phase/gain campaigns and feedback-off brackets. **Pass.** every returned record has `accepted=True`. **Failure.** do not seal or apply a calibration. **Safe to rerun.** yes.


In [ ]:
def assess_gain_linearity(rows, response_field, gain_field='gain',
                          minimum_points=3, minimum_r_squared=0.98):
    """Test proportionality across accepted measured gain points."""
    accepted = [row for row in rows if row.get('accepted', True)]
    if len(accepted) < minimum_points:
        return {'accepted': False, 'reason': 'too few accepted gain points'}
    gain = np.asarray([row[gain_field] for row in accepted], dtype=float)
    response = np.asarray([row[response_field] for row in accepted], dtype=float)
    design = np.column_stack((np.ones(len(gain)), gain))
    # rcond=-1 selects NumPy's legacy default threshold and works on the board.
    coefficients = np.linalg.lstsq(design, response, rcond=-1)[0]
    fitted = design.dot(coefficients)
    residual = response - fitted
    total = np.sum((response - np.mean(response)) ** 2)
    r_squared = 1.0 - np.sum(residual ** 2) / max(total, np.finfo(float).tiny)
    return {'accepted': bool(r_squared >= minimum_r_squared),
            'intercept': float(coefficients[0]),
            'slope': float(coefficients[1]), 'r_squared': float(r_squared),
            'residual_rms': float(np.sqrt(np.mean(residual ** 2))),
            'measured_gain_min': float(np.min(gain)),
            'measured_gain_max': float(np.max(gain))}


def assess_phase_rotation(rows, maximum_rotation_deg=10.0):
    """Reject a feedback phase axis that rotates materially with gain."""
    phases = np.unwrap(np.deg2rad([row['phase_deg'] for row in rows]))
    rotation = float(np.degrees(np.max(phases) - np.min(phases))) if len(phases) else float('inf')
    return {'accepted': bool(len(phases) >= 3 and rotation <= maximum_rotation_deg),
            'rotation_deg': rotation, 'limit_deg': float(maximum_rotation_deg)}


def assess_hysteresis(up_rows, down_rows, value_field,
                      maximum_fraction_of_span=0.05):
    """Compare matched increasing/decreasing gain points."""
    up = {float(row['gain']): float(row[value_field]) for row in up_rows}
    down = {float(row['gain']): float(row[value_field]) for row in down_rows}
    common = sorted([gain for gain in up if gain in down])
    if len(common) < 3:
        return {'accepted': False, 'reason': 'fewer than three paired gains'}
    all_values = [up[g] for g in common] + [down[g] for g in common]
    span = max(all_values) - min(all_values)
    worst = max(abs(up[g] - down[g]) for g in common)
    fraction = worst / max(span, np.finfo(float).tiny)
    return {'accepted': bool(fraction <= maximum_fraction_of_span),
            'paired_gains': common, 'worst_difference': float(worst),
            'fraction_of_span': float(fraction)}


def assess_baseline_recovery(before_fit, after_fit,
                             frequency_limit_linewidths=0.25,
                             q_fraction_limit=0.10):
    """Check that resonator centre and Q recover after feedback is disabled."""
    frequency_change = abs(after_fit['f0_hz'] - before_fit['f0_hz'])
    linewidth = max(abs(before_fit['linewidth_hz']), np.finfo(float).tiny)
    q_change = abs(after_fit['q'] / before_fit['q'] - 1.0)
    return {'accepted': bool(frequency_change <= frequency_limit_linewidths * linewidth and
                             q_change <= q_fraction_limit),
            'frequency_change_hz': float(frequency_change),
            'frequency_change_linewidths': float(frequency_change / linewidth),
            'q_fraction_change': float(q_change)}

def assess_output_commissioning(evidence):
    """Validate OUT1-only commissioning without claiming electrode voltage."""
    required = ('dac_zero_valid', 'out1_scale_valid', 'spectral_purity_valid',
                'coefficient_linearity_valid', 'phase_valid',
                'output_noise_valid', 'clipping_absent', 'shutdown_valid')
    failed = [name for name in required if evidence.get(name) is not True]
    return {'accepted': not failed, 'failed_checks': failed,
            'coil_voltage_directly_measured': False,
            'airgap_transfer_directly_measured': False,
            'effective_loop_status': 'closed_loop_inference_only',
            'out1_measured_vpk': evidence.get('out1_measured_vpk'),
            'declared_passive_attenuation_db': evidence.get('declared_passive_attenuation_db')}


def effective_feedback_evidence(phase_axes, gain_linearity, phase_rotation,
                                hysteresis, recovery_checks):
    """Combine indirect evidence that feedback reached the air-gap coil path."""
    checks = {'four_axes': phase_axes.get('orthogonality_error_deg', 999) <= 10.0,
              'gain_linearity': gain_linearity.get('accepted', False),
              'phase_stability': phase_rotation.get('accepted', False),
              'hysteresis': hysteresis.get('accepted', False),
              'all_recoveries': bool(recovery_checks) and
                                all(row.get('accepted', False) for row in recovery_checks)}
    return {'accepted': all(checks.values()), 'checks': checks,
            'proof_basis': 'coefficient_readback_plus_repeatable_resonator_response',
            'coil_voltage_directly_measured': False,
            'airgap_transfer_directly_measured': False}


---
# 17 — Freeze and print the calibration

**Purpose.** Convert accepted measurements into sealed interpolation envelopes used by the setup
helpers.

**Hardware.** None. **Expected result.** A complete record with evidence fingerprints and explicit
unmeasured coil/air-gap quantities. **Pass.** Noise, baseline, phase, gain, OUT2 and recovery gates
are accepted. **Failure.** The record is refused; never override missing evidence. **Safe to
rerun.** Yes.


In [ ]:
def build_calibration(baseline_result, phase_axes, q_points, shift_points,
                      analyser_comparison, acquisition_calibration,
                      analyser_assessment, output_commissioning,
                      effective_feedback, recovery_results=None,
                      evidence_overrides=None):
    fit = baseline_result['fit']
    evidence = {
        'noise_floor_valid': len(NOISE_CALIBRATION) >= 2,
        'baseline_valid': bool(fit['accepted']),
        'linearity_valid': len(GAIN_RESULTS) >= 6,
        'phase_valid': phase_axes['orthogonality_error_deg'] <= 10.0,
        'gain_linearity_valid': len(q_points) >= 3 and len(shift_points) >= 5,
        'drift_valid': len(BASELINE_HISTORY) >= 3,
        'acquisition_valid': acquisition_calibration.get('accepted', False),
        'analyser_valid': analyser_assessment.get('accepted', False),
        'output_commissioning_valid': output_commissioning.get('accepted', False),
        'effective_feedback_valid': effective_feedback.get('accepted', False),
        'out1_dc_offset_valid': bool(OUT1_DC_COMMISSIONING and
                                     OUT1_DC_COMMISSIONING.get('accepted', False)),
        'out2_monitor_valid': bool(OUT2_MONITOR_CALIBRATION and
                                   OUT2_MONITOR_CALIBRATION.get('accepted', False)),
        'created_epoch_s': time.time(),
        'analyser_comparison': copy.deepcopy(analyser_comparison),
        'analyser_fingerprint': ANALYSER_FINGERPRINT,
        'topology_digest': TOPOLOGY_RECORD['digest'],
        'acquisition_calibration': copy.deepcopy(acquisition_calibration),
        'analyser_assessment': copy.deepcopy(analyser_assessment),
        'output_commissioning': copy.deepcopy(output_commissioning),
        'effective_feedback': copy.deepcopy(effective_feedback),
        'feedback_off_recoveries': copy.deepcopy(recovery_results or []),
    }
    if evidence_overrides:
        evidence.update(evidence_overrides)
    safety = {
        'max_k': ELECTRODE_MAX_K,
        'topology_digest': TOPOLOGY_RECORD['digest'],
        'return_attenuation_db_nominal': TOPOLOGY['return_attenuation_db_nominal'],
        'input_attenuation_db': TOPOLOGY['input_attenuation_db'],
        'sense_capacitance_f': TOPOLOGY['sense_capacitance_f'],
        'in1_termination_ohm': TOPOLOGY['in1_termination_ohm'],
        'analyser_impedance_ohm': ANALYSER['input_impedance_ohm'],
        'airgap_geometry_id': TOPOLOGY['airgap_geometry_id'],
        'coil_voltage_directly_measured': False,
        'airgap_transfer_directly_measured': False,
    }
    calibration = make_calibration(
        {'f0_hz': fit['f0_hz'], 'q0': fit['q'],
         'linewidth_hz': fit['linewidth_hz']},
        q_points, shift_points, phase_axes, evidence, safety, CURRENT_BUILD_ID)
    printable = copy.deepcopy(calibration)
    print(json.dumps(printable, indent=2, default=str))
    return calibration

# CALIBRATION = build_calibration(
#     baseline, PHASE_AXES, q_points, shift_points, analyser_comparison,
#     acquisition_calibration, analyser_assessment, output_commissioning,
#     effective_feedback, recovery_results)


---
# 18 — Manual recommendations

**Purpose.** Resolve a requested Q or shift using accepted measured points and interpolation only.

**Hardware.** None. **Expected result.** A sealed gain/phase setup with prediction, uncertainty and
interpolation bracket. **Pass.** The request lies inside a verified envelope. **Failure.**
Conflicting targets, stale calibration or extrapolation are refused. **Safe to rerun.** Yes.


In [ ]:
def print_setup(setup):
    print(json.dumps(setup, indent=2, default=str))
    return setup

def recommend_q(target_q=None, q_ratio=None, target_k=None, gain=None):
    return print_setup(feedback_q_setup(
        CALIBRATION, target_q=target_q, q_ratio=q_ratio,
        target_k=target_k, gain=gain))

def recommend_shift(shift_hz=None, shift_linewidths=None,
                    signed_gain=None, target_k=None, direction=None):
    return print_setup(feedback_shift_setup(
        CALIBRATION, shift_hz=shift_hz, shift_linewidths=shift_linewidths,
        signed_gain=signed_gain, target_k=target_k, direction=direction))

# Examples:
# proposed_setup = recommend_q(q_ratio=1.10)
# proposed_setup = recommend_shift(shift_linewidths=1.0)


---
# 19 — Guided selection

**Purpose.** Present conservative measured choices and explain their expected resonator effect.

**Hardware.** None. **Expected result.** A candidate setup for review, not an applied setting.
**Pass.** The candidate belongs to a sealed envelope. **Failure.** Missing direction or out-of-range
target. **Safe to rerun.** Yes.


In [ ]:
def guide_selection(goal, value=None):
    goals = ('q_enhancement', 'q_reduction', 'positive_shift', 'negative_shift')
    if goal not in goals:
        raise ValueError('goal must be one of: ' + ', '.join(goals))
    if goal == 'q_enhancement':
        setup = feedback_q_setup(CALIBRATION, q_ratio=value) if value else feedback_q_setup(CALIBRATION)
        consequence = 'Higher resonator Q means reduced effective damping and a narrower, taller noise peak.'
    elif goal == 'q_reduction':
        raise SetupError('Q-reduction recommendations require a separately calibrated degenerative envelope')
    else:
        direction = 1 if goal == 'positive_shift' else -1
        setup = (feedback_shift_setup(CALIBRATION, shift_linewidths=direction * abs(value))
                 if value else feedback_shift_setup(CALIBRATION)[
                     'positive' if direction > 0 else 'negative'])
        consequence = 'The resonance centre shifts while the calibrated reactive axis should leave Q nearly unchanged.'
    print(consequence)
    return print_setup(setup)


---
# 20 — Apply one reviewed setup

**Purpose.** Revalidate a sealed setup immediately before enabling OUT1 and acquire a short
validation spectrum.

**Hardware.** Yes, only after the final `APPLY FEEDBACK` confirmation. **Expected result.** Correct
coefficient readback, healthy OUT2, unclipped spectrum and measured centre compatible with the
prediction. **Pass.** The validated setting remains active until `feedback_off()`. **Failure.** Any
exception, interruption, drift, monitor fault or disagreement invokes `full_shutdown()`. **Safe to
rerun.** Only after a fresh feedback-off baseline.


In [ ]:
def feedback_apply(setup, confirmation=None):
    try:
        feedback_off()
        validate_setup(setup, CALIBRATION, current_build_id='0x%08X' % acq.build_id(),
                       max_age_s=CALIBRATION_MAX_AGE_S, now_s=time.time())
        report_now = configuration_report()
        if OUT1_DC_COMMISSIONING is None or not OUT1_DC_COMMISSIONING.get('accepted', False):
            raise SetupError('OUT1 DC/transient commissioning is missing')
        if OUT2_MONITOR_CALIBRATION is None or not OUT2_MONITOR_CALIBRATION.get('accepted', False):
            raise SetupError('OUT2 campaign monitor is not commissioned')
        monitor_status = out2_monitor_status()
        if monitor_status['source_code'] != 1 or monitor_status['shift'] != 0 or monitor_status['clipped']:
            raise SetupError('OUT2 unity monitor is not healthy')
        if _digest(TOPOLOGY) != _digest(TOPOLOGY_RECORD['payload']['config']):
            raise SetupError('topology changed since calibration')
        if analyser_configuration_fingerprint(ANALYSER) != ANALYSER_FINGERPRINT:
            raise SetupError('analyser or cable settings changed')
        baseline_now = measure_baseline('pre_apply')
        fit_now = baseline_now['fits']['none']
        baseline_cal = CALIBRATION['baseline']
        if abs(fit_now['f0_hz'] - baseline_cal['f0_hz']) > 0.25 * baseline_cal['linewidth_hz']:
            raise SetupError('current resonator baseline has drifted outside allowance')
        resolved = setup['resolved']
        gain = resolved.get('gain', abs(resolved.get('signed_gain', 0.0)))
        phase = resolved['phase_deg']
        live_codes, _ = sc.acquire(dec=1, n=4096, chan='A')
        live_volts = live_codes * ADC_RAW_VOLTS_PER_COUNT
        live_dc_v = float(np.mean(live_volts))
        predicted_feedback_dc_bound_v = abs(live_dc_v) * abs(gain) * np.sqrt(2.0)
        if predicted_feedback_dc_bound_v >= OUT1_DC_OFFSET_LIMIT_V:
            raise SetupError(
                'Current IN1 mean %.6g V and gain %.6g imply a conservative '
                'OUT1 feedback DC bound %.6g V, above the %.6g V limit. The '
                'fractional-delay phase shifter passes DC; identify the ADC/chain '
                'offset source or choose a smaller measured gain.' %
                (live_dc_v, gain, predicted_feedback_dc_bound_v,
                 OUT1_DC_OFFSET_LIMIT_V))
        proposed = {'OUT1_feedback_gain': gain, 'phase_deg': phase,
                    'predicted_f0_hz': resolved['predicted_f0_hz'],
                    'predicted_k': resolved['k'],
                    'measured_in1_dc_v': live_dc_v,
                    'predicted_out1_dc_bound_v': predicted_feedback_dc_bound_v,
                    'resonator_state': 'passive_noise'}
        print('PROPOSED ACTION\n' + json.dumps(proposed, indent=2))
        answer = confirmation if confirmation is not None else input('Type APPLY FEEDBACK exactly: ')
        if answer != 'APPLY FEEDBACK':
            raise SetupError('manual confirmation rejected')
        input_peak_v = sc.peak_frac(4096) * RP_LIMITS['input_lv_full_scale_vpk']
        predicted_out1_vpk = (input_peak_v / RP_LIMITS['input_lv_full_scale_vpk'] *
                              abs(gain) * RP_LIMITS['output_full_scale_vpk_into_50ohm'])
        if input_peak_v >= ADC_OPERATING_GUARD_VPK:
            raise SetupError('IN1 live peak is too close to full scale')
        if predicted_out1_vpk >= OUT1_OPERATING_GUARD_VPK:
            raise SetupError('conservative OUT1 bound exceeds the operating guard')
        ps.config_delay(CENTRE_HZ)
        coefficients = ps.set(gain, phase)
        if (ps.readback()['ci'], ps.readback()['cq']) != tuple(coefficients):
            raise SetupError('coefficient readback failed')
        validation_plan = acquisition_plan(
            baseline_cal['f0_hz'], max(resolved.get('q', baseline_cal['q0']), 1.0),
            resolved.get('shift_hz', 0.0), centre_on_shift=True)
        validation_records = acquire_passive_records(
            plan=validation_plan, averages=max(8, PASSIVE_AVERAGES // 4),
)
        if sc.peak_frac(4096) * RP_LIMITS['input_lv_full_scale_vpk'] >= ADC_OPERATING_GUARD_VPK:
            raise SetupError('IN1 live peak exceeded the guard during validation')
        validation_spectrum = make_passive_spectrum(validation_records,
                                                    plan=validation_plan)
        baseline_floor = float(np.median(baseline_now['spectrum']['psd_v2_per_hz']))
        baseline_reference = np.interp(
            validation_spectrum['frequency_hz'],
            baseline_now['spectrum']['frequency_hz'],
            baseline_now['spectrum']['psd_v2_per_hz'],
            left=baseline_floor, right=baseline_floor)
        instability = detect_self_oscillation(
            validation_spectrum['frequency_hz'], validation_spectrum['psd_v2_per_hz'],
            baseline_reference)
        validation_q_window = (max(20.0, validation_plan['q_estimate']/5.0),
                               min(1.0e6, validation_plan['q_estimate']*5.0))
        validation_fit = fit_one_spectrum(
            validation_spectrum, discovery=False, q_range=validation_q_window)
        tolerance = max(3.0 * validation_fit['f0_uncertainty_hz'],
                        0.20 * baseline_cal['linewidth_hz'])
        if instability['unsafe'] or not validation_fit['accepted']:
            raise SetupError('validation detected instability or an invalid fit')
        if abs(validation_fit['f0_hz'] - resolved['predicted_f0_hz']) > tolerance:
            raise SetupError('validation disagrees with predicted centre frequency')
        print('PASS: feedback applied and validated. Use feedback_off() when finished.')
        return {'setup_digest': setup['digest'], 'coefficients': coefficients,
                'validation_fit': validation_fit, 'instability': instability}
    except BaseException:
        full_shutdown()
        raise


---
# 21 — Export the campaign

**Purpose.** Save configuration, calibration and compact results to JSON, with raw spectra in NPZ.

**Hardware.** None. **Expected result.** One JSON and one compressed NPZ file. **Pass.** Both reopen
successfully. **Failure.** Filesystem/path error; measurement state is unchanged. **Safe to
rerun.** Yes with a new output name.


In [ ]:
def export_campaign(output_stem):
    stem = Path(output_stem)
    metadata = {
        'campaign': CAMPAIGN_LABEL, 'topology': TOPOLOGY,
        'topology_digest': TOPOLOGY_RECORD['digest'],
        'analyser': ANALYSER, 'analyser_fingerprint': ANALYSER_FINGERPRINT,
        'build_id': CURRENT_BUILD_ID, 'calibration': CALIBRATION,
        'coil_voltage_directly_measured': False,
        'airgap_transfer_directly_measured': False,
        'return_attenuation_db_nominal': 30.0,
        'input_attenuation_db': 0.0,
        'sense_capacitance_f': 2.2e-12,
        'effective_loop_status': 'closed_loop_inference_only',
        'out1_dc_commissioning': OUT1_DC_COMMISSIONING,
        'out2_monitor_calibration': OUT2_MONITOR_CALIBRATION,
        'phase_results_summary': [
            {'gain': row['gain'], 'phase_deg': row['phase_deg'],
             'f0_hz': row['fit']['f0_hz'], 'q': row['fit']['q'],
             'drift_hz': row.get('drift_hz')} for row in PHASE_RESULTS],
        'gain_results_summary': [
            {'gain': row['gain'], 'phase_deg': row['phase_deg'],
             'axis': row.get('axis'), 'f0_hz': row['fit']['f0_hz'],
             'q': row['fit']['q']} for row in GAIN_RESULTS],
        'exported_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    stem.with_suffix('.json').write_text(json.dumps(metadata, indent=2, default=str),
                                         encoding='utf-8')
    arrays = {}
    for number, item in enumerate(BASELINE_HISTORY):
        arrays['baseline_%03d_frequency_hz' % number] = item['spectrum']['frequency_hz']
        arrays['baseline_%03d_psd' % number] = item['spectrum']['psd_v2_per_hz']
    for prefix, rows in (('phase', PHASE_RESULTS), ('gain', GAIN_RESULTS)):
        for number, item in enumerate(rows):
            arrays['%s_%03d_frequency_hz' % (prefix, number)] = item['spectrum']['frequency_hz']
            arrays['%s_%03d_psd' % (prefix, number)] = item['spectrum']['psd_v2_per_hz']
    np.savez_compressed(str(stem.with_suffix('.npz')), **arrays)
    print('Wrote', stem.with_suffix('.json'), 'and', stem.with_suffix('.npz'))


---
# Acceptance summary

A successful software run is not sufficient. Acceptance requires repeatable noise baselines,
analyser agreement, a verified linear interval, all four phase axes, compatible K estimates,
three Q settings, three shifts in each direction, and feedback-off recovery after every campaign.

A hardware, pad, cable, amplifier or air-gap change creates a new campaign: rerun live safety,
noise baselines, OUT2 comparison, phase calibration and low-to-high gain commissioning.
